 <b><i> analysis_teu_amoc </i></b>

Created by Eduardo Alastrue de Asenjo on 2025-02-21

- Purpose: analyse AMOC & temperature data from new hosing simulations
- Method: 
- Comments: use env_24 to work with regionmask

In [ ]:
import os
current_dir = os.getcwd()
if 'm300940' in current_dir:
    is_felix = True
else:
    is_felix = False
user_name = "m300940" if is_felix else "m300817"

In [ ]:
# Load modules, works with kernel "env_24" in DKRZ's JupyterHub
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.lines as mlines
from matplotlib.colors import BoundaryNorm
from matplotlib.patches import Rectangle
from matplotlib.gridspec import GridSpec
from matplotlib.lines import Line2D
import matplotlib.path as mpath
import seaborn as sns
sns.set(rc={'figure.figsize':(11.7,8.27)},font_scale=1.5) # 'font.size': 12
sns.set_style("white")
plt.rcParams['xtick.bottom'] = True
plt.rcParams['ytick.left'] = True
import cartopy
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import cartopy.io.shapereader as shpreader
from shapely.geometry import LineString
from shapely.geometry import Polygon
import re
import math as math
import netCDF4 as nc
from scipy import stats
from random import choices
import copy
import dask
import dask.array as da
import xarray as xr
import pandas as pd
import datetime
import cftime
from tqdm import tqdm
import glob
import pickle
import cdo # Import Cdo-py
if is_felix:
    cdo = cdo.Cdo()
else:
    cdo = cdo.Cdo(tempdir="/scratch/m/m300817/tmp/cdo-py") # doesn't work with "My Kernel"! 
import statsmodels.api as sm 
import xclim
import eofs
if 'ESMFMKFILE' not in os.environ: # RTD doesn't activate the env, and esmpy depends on a env var set there, see conda-forge/esmf-feedstock#91 and readthedocs/readthedocs.org#4067
    os.environ['ESMFMKFILE'] = '/work/uo1075/m300817/conda/envs/env_24/lib/esmf.mk'
import xesmf as xe
import nc_time_axis
import cf_xarray as cfxr
import regionmask
import warnings
import json
warnings.filterwarnings("ignore", category=RuntimeWarning)

In [ ]:
import importlib
import functions
importlib.reload(functions)
hosing_colors = functions.hosing_colors
hosing_names = functions.hosing_names

In [ ]:
# local cluster
from dask.distributed import Client
client = Client(memory_limit=None, threads_per_worker=1, n_workers=18)

In [ ]:
def weighted_area_lat(ds):
    """
    Calculate the area-weighted temperature over its domain. 
    In a regular latitude/ longitude grid the grid cell area decreases towards the pole.
    We can use the cosine of the latitude as proxy for the grid cell area.
    IMPORTANT: after applying function, do .mean('lat') before 'lon'
    Taken from https://docs.xarray.dev/en/stable/examples/area_weighted_temperature.html
    
    Parameters:
    ds (xr.Dataset): Dataset containing latitude data.
    
    Returns:
    xr.Dataset: Dataset with weighted area.
    """
    weights = np.cos(np.deg2rad(ds.lat))
    weights.name = "weights"
    return ds.weighted(weights)


In [ ]:
def convert360_180(_ds):
    """
    convert longitude from 0-360 to -180 -- 180 deg
    """
    # check if already 
    attrs = _ds['lon'].attrs
    if _ds['lon'].min() >= 0:
        with xr.set_options(keep_attrs=True): 
            _ds.coords['lon'] = (_ds['lon'] + 180) % 360 - 180
        _ds = _ds.sortby('lon')
    return _ds

In [ ]:
def weighted_mon_to_year_mean(ds, var):
    """
    weight by days in each month when doing annual mean from monthly values in xarray
    taken from https://ncar.github.io/esds/posts/2021/yearly-averages-xarray/
    """
    month_length = ds.time.dt.days_in_month # Determine the month length

    wgts = month_length.groupby("time.year") / month_length.groupby("time.year").sum() # Calculate the weights

    np.testing.assert_allclose(wgts.groupby("time.year").sum(xr.ALL_DIMS), 1.0) # Make sure the weights in each year add up to 1

    obs = ds[var] # Subset our dataset for our variable

    cond = obs.isnull() # Setup our masking for nan values
    ones = xr.where(cond, 0.0, 1.0)
    
    obs_sum = (obs * wgts).resample(time="YS").sum(dim="time") # Calculate the numerator
    ones_out = (ones * wgts).resample(time="YS").sum(dim="time") # Calculate the denominator

    return obs_sum / ones_out # Return the weighted average

In [ ]:
def add_square(ax, lon_min, lon_max, lat_min, lat_max, colour='white'):
    rectangle = Polygon([(lon_min, lat_min), (lon_min, lat_max), (lon_max, lat_max), (lon_max, lat_min)])
    ax.add_geometries([rectangle], crs=ccrs.PlateCarree(), facecolor=colour, edgecolor=colour, zorder=4)

In [ ]:
ssp_col = {'ssp126': (23/255,60/255,102/255),
           'ssp245': (247/255,148/255,32/255),
           'ssp370': (231/255,29/255,37/255)
          }

# Load files

MPI-ESM hosing 

In [ ]:
outpath = f"/work/uo1075/m300817/teu_amoc/data/ssphos/" 

In [ ]:
ssphos_amoc_yr = xr.open_mfdataset(outpath+"ssphos_amoc_yr.nc", use_cftime=True)
ssphos_amoc26_deep_yr = xr.open_mfdataset(outpath+"ssphos_amoc26_deep_yr.nc", use_cftime=True)

In [ ]:
ssphos_tas_mon = xr.open_dataarray(outpath+"ssphos_tas_mon.nc", use_cftime=True).to_dataset(name='tas')
ssphos_tas_yr = xr.open_dataarray(outpath+"ssphos_tas_yr.nc", use_cftime=True).to_dataset(name='tas')
ssphos_tas_djf = xr.open_dataarray(outpath+"ssphos_tas_djf.nc", use_cftime=True).to_dataset(name='tas')
ssphos_tas_jja = xr.open_dataarray(outpath+"ssphos_tas_jja.nc", use_cftime=True).to_dataset(name='tas')
ssphos_tas_eur_yr = xr.open_dataarray(outpath+"ssphos_tas_eur_yr.nc", use_cftime=True).to_dataset(name='tas')
ssphos_tas_eur_djf = xr.open_dataarray(outpath+"ssphos_tas_eur_djf.nc", use_cftime=True).to_dataset(name='tas')
ssphos_tas_eur_jja = xr.open_dataarray(outpath+"ssphos_tas_eur_jja.nc", use_cftime=True).to_dataset(name='tas')

MPI-ESM GE

In [ ]:
outpath = f"/work/uo1075/m300817/teu_amoc/data/MPI-GE/"

his_amoc_yr = xr.open_dataarray(outpath+"his_amoc26_yr.nc", use_cftime=True).to_dataset(name='amoc')
his_amoc_yr_ensmean = his_amoc_yr.mean("realiz")
his_amoc_yr_ensstd = his_amoc_yr.std("realiz")

ssp_amoc_yr = xr.open_dataarray(outpath+"ssp_amoc26_yr.nc", use_cftime=True).to_dataset(name='amoc')
ssp_amoc_yr_ensmean = ssp_amoc_yr.mean("realiz")
ssp_amoc_yr_ensstd = ssp_amoc_yr.std("realiz")

In [ ]:
for sce in ["ssp126", "ssp370"]:
    print(1-ssp_amoc_yr.sel(scenar=sce).mean("realiz").isel(time=slice(-10, None)).mean("time").amoc.values/19.0)

In [ ]:
his_tas_yr = xr.open_dataarray(outpath+"his_tas_yr.nc", use_cftime=True).to_dataset(name='tas')
his_tas_yr_ensmean = his_tas_yr.mean("realiz")
his_tas_djf = xr.open_dataset(outpath+"his_tas_djf.nc", use_cftime=True)
his_tas_jja = xr.open_dataset(outpath+"his_tas_jja.nc", use_cftime=True)
his_tas_eur_yr = xr.open_dataarray(outpath+"his_tas_eur_yr.nc", use_cftime=True).to_dataset(name='tas')
his_tas_eur_yr_ensmean = his_tas_eur_yr.mean("realiz")
his_tas_eur_djf = xr.open_dataset(outpath+"his_tas_eur_djf.nc", use_cftime=True)
his_tas_eur_jja = xr.open_dataset(outpath+"his_tas_eur_jja.nc", use_cftime=True)

ssp_tas_yr = xr.open_dataarray(outpath+"ssp_tas_yr.nc", use_cftime=True).to_dataset(name='tas')
ssp_tas_yr_ensmean = ssp_tas_yr.mean("realiz")
ssp_tas_djf = xr.open_dataset(outpath+"ssp_tas_djf.nc", use_cftime=True)
ssp_tas_jja = xr.open_dataset(outpath+"ssp_tas_jja.nc", use_cftime=True)
ssp_tas_eur_yr = xr.open_dataarray(outpath+"ssp_tas_eur_yr.nc", use_cftime=True).to_dataset(name='tas')
ssp_tas_eur_yr_ensmean = ssp_tas_eur_yr.mean("realiz")
ssp_tas_eur_djf = xr.open_dataset(outpath+"ssp_tas_eur_djf.nc", use_cftime=True)
ssp_tas_eur_jja = xr.open_dataset(outpath+"ssp_tas_eur_jja.nc", use_cftime=True)

CMIP6

In [ ]:
# AMOC

In [ ]:
# historical
with open(f"/work/uo1075/m300817/teu_amoc/data/CMIP6/historical/cmip6_historical_amoc_dict.json", 'r') as json_file:
    cmip6_historical_amoc_dict = json.load(json_file)
with open(f"/work/uo1075/m300817/teu_amoc/data/CMIP6/historical/cmip6_historical_amoc_msftyz_dict.json", 'r') as json_file:
    cmip6_historical_amoc_msftyz_dict = json.load(json_file)
cmip6_historical_amoc_dict = cmip6_historical_amoc_dict | cmip6_historical_amoc_msftyz_dict
historical_amoc_cmip6 = {}
countmodhis, countsimhis = 0,0
for model in cmip6_historical_amoc_dict.keys():
    historical_amoc_cmip6[model] = {}
    countmodhis = countmodhis+1
    for rea in cmip6_historical_amoc_dict[model]:
        historical_amoc_cmip6[model][rea]= xr.open_dataarray(
            f"/work/uo1075/m300817/teu_amoc/data/CMIP6/historical/{model}/{model}_{rea}_amoc26_yr.nc", 
            use_cftime=True).to_dataset(name='amoc')
        countsimhis = countsimhis+1
print(f"{countsimhis} simulations from {countmodhis} models loaded")

In [ ]:
# ssp126
with open(f"/work/uo1075/m300817/teu_amoc/data/CMIP6/ssp126/cmip6_ssp126_amoc_dict.json", 'r') as json_file:
    cmip6_ssp126_amoc_dict = json.load(json_file)
with open(f"/work/uo1075/m300817/teu_amoc/data/CMIP6/ssp126/cmip6_ssp126_amoc_msftyz_dict.json", 'r') as json_file:
    cmip6_ssp126_amoc_msftyz_dict = json.load(json_file)
cmip6_ssp126_amoc_dict = cmip6_ssp126_amoc_dict | cmip6_ssp126_amoc_msftyz_dict

ssp126_amoc_cmip6 = {}
countmod126, countsim126 = 0,0
for model in cmip6_ssp126_amoc_dict.keys():
    ssp126_amoc_cmip6[model] = {}
    countmod126 = countmod126+1
    for rea in cmip6_ssp126_amoc_dict[model]:
        ssp126_amoc_cmip6[model][rea]= xr.open_dataarray(
            f"/work/uo1075/m300817/teu_amoc/data/CMIP6/ssp126/{model}/{model}_{rea}_amoc26_yr.nc", 
            use_cftime=True).to_dataset(name='amoc')
        countsim126 = countsim126+1
print(f"{countsim126} simulations from {countmod126} models loaded")

In [ ]:
# ssp245
with open(f"/work/uo1075/m300817/teu_amoc/data/CMIP6/ssp245/cmip6_ssp245_amoc_dict.json", 'r') as json_file:
    cmip6_ssp245_amoc_dict = json.load(json_file)
with open(f"/work/uo1075/m300817/teu_amoc/data/CMIP6/ssp245/cmip6_ssp245_amoc_msftyz_dict.json", 'r') as json_file:
    cmip6_ssp245_amoc_msftyz_dict = json.load(json_file)
cmip6_ssp245_amoc_dict = cmip6_ssp245_amoc_dict | cmip6_ssp245_amoc_msftyz_dict
    
ssp245_amoc_cmip6 = {}
countmod245, countsim245 = 0,0
for model in cmip6_ssp245_amoc_dict.keys():
    ssp245_amoc_cmip6[model] = {}
    countmod245 = countmod245+1
    for rea in cmip6_ssp245_amoc_dict[model]:
        ssp245_amoc_cmip6[model][rea]= xr.open_dataarray(
            f"/work/uo1075/m300817/teu_amoc/data/CMIP6/ssp245/{model}/{model}_{rea}_amoc26_yr.nc", 
            use_cftime=True).to_dataset(name='amoc')
        countsim245= countsim245+1
print(f"{countsim245} simulations from {countmod245} models loaded")

In [ ]:
# ssp370
with open(f"/work/uo1075/m300817/teu_amoc/data/CMIP6/ssp370/cmip6_ssp370_amoc_dict.json", 'r') as json_file:
    cmip6_ssp370_amoc_dict = json.load(json_file)
with open(f"/work/uo1075/m300817/teu_amoc/data/CMIP6/ssp370/cmip6_ssp370_amoc_msftyz_dict.json", 'r') as json_file:
    cmip6_ssp370_amoc_msftyz_dict = json.load(json_file)
cmip6_ssp370_amoc_dict = cmip6_ssp370_amoc_dict | cmip6_ssp370_amoc_msftyz_dict

ssp370_amoc_cmip6 = {}
countmod370, countsim370 = 0,0
for model in cmip6_ssp370_amoc_dict.keys():
    ssp370_amoc_cmip6[model] = {}
    countmod370= countmod370+1
    for rea in cmip6_ssp370_amoc_dict[model]:
        ssp370_amoc_cmip6[model][rea]= xr.open_dataarray(
            f"/work/uo1075/m300817/teu_amoc/data/CMIP6/ssp370/{model}/{model}_{rea}_amoc26_yr.nc", 
            use_cftime=True).to_dataset(name='amoc')
        countsim370 = countsim370+1
print(f"{countsim370} simulations from {countmod370} models loaded")

In [ ]:
# TAS

In [ ]:
# historical
with open(f"/work/uo1075/m300817/teu_amoc/data/CMIP6/historical/cmip6_historical_tas_dict.json", 'r') as json_file:
    cmip6_historical_tas_dict = json.load(json_file)
with open(f"/work/uo1075/m300817/teu_amoc/data/CMIP6/historical/cmip6_historical_tas_msftyz_dict.json", 'r') as json_file:
    cmip6_historical_tas_msftyz_dict = json.load(json_file)
cmip6_historical_tas_dict = cmip6_historical_tas_dict | cmip6_historical_tas_msftyz_dict

historical_tas_cmip6, historical_tas_eur_cmip6 = {}, {}
countmod, countsim = 0,0
for model in cmip6_historical_tas_dict.keys():
    historical_tas_cmip6[model], historical_tas_eur_cmip6[model] = {}, {}
    countmod = countmod+1
    for rea in cmip6_historical_tas_dict[model]:
        historical_tas_cmip6[model][rea]= xr.open_dataarray(
            f"/work/uo1075/m300817/teu_amoc/data/CMIP6/historical/{model}/{model}_{rea}_tas_yr.nc", 
            use_cftime=True).to_dataset(name='tas')
        mask_eur = regionmask.defined_regions.ar6.land[16, 17, 18, 19].mask(historical_tas_cmip6[model][rea])  # Europe
        mask_land = regionmask.defined_regions.natural_earth_v5_0_0.land_110.mask(historical_tas_cmip6[model][rea]) # Natural earth masks
        historical_tas_eur_cmip6[model][rea] = historical_tas_cmip6[model][rea].where(mask_eur>15, drop=True).where(mask_land==0, drop=True)
        countsim = countsim+1
print(f"{countsim} simulations from {countmod} models loaded")

In [ ]:
# ssp126
with open(f"/work/uo1075/m300817/teu_amoc/data/CMIP6/ssp126/cmip6_ssp126_tas_dict.json", 'r') as json_file:
    cmip6_ssp126_tas_dict = json.load(json_file)
with open(f"/work/uo1075/m300817/teu_amoc/data/CMIP6/ssp126/cmip6_ssp126_tas_msftyz_dict.json", 'r') as json_file:
    cmip6_ssp126_tas_msftyz_dict = json.load(json_file)
cmip6_ssp126_tas_dict = cmip6_ssp126_tas_dict | cmip6_ssp126_tas_msftyz_dict

ssp126_tas_cmip6, ssp126_tas_eur_cmip6 = {}, {}
countmod126, countsim126 = 0,0
for model in cmip6_ssp126_tas_dict.keys():
    ssp126_tas_cmip6[model], ssp126_tas_eur_cmip6[model] = {}, {}
    countmod126 = countmod126+1
    for rea in cmip6_ssp126_tas_dict[model]:
        ssp126_tas_cmip6[model][rea]= xr.open_dataarray(
            f"/work/uo1075/m300817/teu_amoc/data/CMIP6/ssp126/{model}/{model}_{rea}_tas_yr.nc", 
            use_cftime=True).to_dataset(name='tas')
        mask_eur = regionmask.defined_regions.ar6.land[16, 17, 18, 19].mask(ssp126_tas_cmip6[model][rea])  # Europe
        mask_land = regionmask.defined_regions.natural_earth_v5_0_0.land_110.mask(ssp126_tas_cmip6[model][rea]) # Natural earth masks
        ssp126_tas_eur_cmip6[model][rea] = ssp126_tas_cmip6[model][rea].where(mask_eur>15, drop=True).where(mask_land==0, drop=True)
        countsim126 = countsim126+1
print(f"{countsim126} simulations from {countmod126} models loaded")

In [ ]:
# ssp245
with open(f"/work/uo1075/m300817/teu_amoc/data/CMIP6/ssp245/cmip6_ssp245_tas_dict.json", 'r') as json_file:
    cmip6_ssp245_tas_dict = json.load(json_file)
with open(f"/work/uo1075/m300817/teu_amoc/data/CMIP6/ssp245/cmip6_ssp245_tas_msftyz_dict.json", 'r') as json_file:
    cmip6_ssp245_tas_msftyz_dict = json.load(json_file)
cmip6_ssp245_tas_dict = cmip6_ssp245_tas_dict | cmip6_ssp245_tas_msftyz_dict

ssp245_tas_cmip6, ssp245_tas_eur_cmip6 = {}, {}
countmod245, countsim245 = 0,0
for model in cmip6_ssp245_tas_dict.keys():
    ssp245_tas_cmip6[model], ssp245_tas_eur_cmip6[model] = {}, {}
    countmod245 = countmod245+1
    for rea in cmip6_ssp245_tas_dict[model]:
        ssp245_tas_cmip6[model][rea]= xr.open_dataarray(
            f"/work/uo1075/m300817/teu_amoc/data/CMIP6/ssp245/{model}/{model}_{rea}_tas_yr.nc", 
            use_cftime=True).to_dataset(name='tas')
        mask_eur = regionmask.defined_regions.ar6.land[16, 17, 18, 19].mask(ssp245_tas_cmip6[model][rea])  # Europe
        mask_land = regionmask.defined_regions.natural_earth_v5_0_0.land_110.mask(ssp245_tas_cmip6[model][rea]) # Natural earth masks
        ssp245_tas_eur_cmip6[model][rea] = ssp245_tas_cmip6[model][rea].where(mask_eur>15, drop=True).where(mask_land==0, drop=True)
        countsim245 = countsim245+1
print(f"{countsim245} simulations from {countmod245} models loaded")

In [ ]:
# ssp370
with open(f"/work/uo1075/m300817/teu_amoc/data/CMIP6/ssp370/cmip6_ssp370_tas_dict.json", 'r') as json_file:
    cmip6_ssp370_tas_dict = json.load(json_file)
with open(f"/work/uo1075/m300817/teu_amoc/data/CMIP6/ssp370/cmip6_ssp370_tas_msftyz_dict.json", 'r') as json_file:
    cmip6_ssp370_tas_msftyz_dict = json.load(json_file)
cmip6_ssp370_tas_dict = cmip6_ssp370_tas_dict | cmip6_ssp370_tas_msftyz_dict

ssp370_tas_cmip6, ssp370_tas_eur_cmip6 = {}, {}
countmod370, countsim370 = 0,0
for model in cmip6_ssp370_tas_dict.keys():
    ssp370_tas_cmip6[model], ssp370_tas_eur_cmip6[model] = {}, {}
    countmod370 = countmod370+1
    for rea in cmip6_ssp370_tas_dict[model]:
        ssp370_tas_cmip6[model][rea]= xr.open_dataarray(
            f"/work/uo1075/m300817/teu_amoc/data/CMIP6/ssp370/{model}/{model}_{rea}_tas_yr.nc", 
            use_cftime=True).to_dataset(name='tas')
        mask_eur = regionmask.defined_regions.ar6.land[16, 17, 18, 19].mask(ssp370_tas_cmip6[model][rea])  # Europe
        mask_land = regionmask.defined_regions.natural_earth_v5_0_0.land_110.mask(ssp370_tas_cmip6[model][rea]) # Natural earth masks
        ssp370_tas_eur_cmip6[model][rea] = ssp370_tas_cmip6[model][rea].where(mask_eur>15, drop=True).where(mask_land==0, drop=True)
        countsim370 = countsim370+1
print(f"{countsim370} simulations from {countmod370} models loaded")

add uploaded data not in dict

In [ ]:
# CESM2

In [ ]:
ssp126_amoc_cmip6['cesm2'] = {}
for rea in ['r4i1p1f1', 'r10i1p1f1', 'r11i1p1f1']:
    ssp126_amoc_cmip6['cesm2'][rea] = xr.open_dataarray(f"/work/uo1075/m300817/teu_amoc/data/CMIP6/ssp126/cesm2/cesm2_{rea}_amoc26_yr.nc", use_cftime=True).to_dataset(name='amoc')

ssp370_amoc_cmip6['cesm2'] = {}
for rea in ['r4i1p1f1', 'r5i1p1f1', 'r6i1p1f1', 'r10i1p1f1', 'r11i1p1f1']:
    ssp370_amoc_cmip6['cesm2'][rea] = xr.open_dataarray(f"/work/uo1075/m300817/teu_amoc/data/CMIP6/ssp370/cesm2/cesm2_{rea}_amoc26_yr.nc", use_cftime=True).to_dataset(name='amoc')

In [ ]:
ssp126_tas_cmip6['cesm2'], ssp126_tas_eur_cmip6['cesm2'] = {}, {}
for rea in ['r4i1p1f1', 'r10i1p1f1', 'r11i1p1f1']:
    ssp126_tas_cmip6['cesm2'][rea] = xr.open_dataarray(f"/work/uo1075/m300817/teu_amoc/data/CMIP6/ssp126/cesm2/cesm2_{rea}_tas_yr.nc", use_cftime=True).to_dataset(name='tas')
    mask_eur = regionmask.defined_regions.ar6.land[16, 17, 18, 19].mask(ssp126_tas_cmip6['cesm2'][rea])  # Europe
    mask_land = regionmask.defined_regions.natural_earth_v5_0_0.land_110.mask(ssp126_tas_cmip6['cesm2'][rea]) # Natural earth masks
    ssp126_tas_eur_cmip6['cesm2'][rea] = ssp126_tas_cmip6['cesm2'][rea].where(mask_eur>15, drop=True).where(mask_land==0, drop=True)

ssp370_tas_cmip6['cesm2'], ssp370_tas_eur_cmip6['cesm2'] = {}, {}
for rea in ['r4i1p1f1', 'r5i1p1f1', 'r6i1p1f1', 'r10i1p1f1', 'r11i1p1f1']:
    ssp370_tas_cmip6['cesm2'][rea] = xr.open_dataarray(f"/work/uo1075/m300817/teu_amoc/data/CMIP6/ssp370/cesm2/cesm2_{rea}_tas_yr.nc", use_cftime=True).to_dataset(name='tas')
    mask_eur = regionmask.defined_regions.ar6.land[16, 17, 18, 19].mask(ssp370_tas_cmip6['cesm2'][rea])  # Europe
    mask_land = regionmask.defined_regions.natural_earth_v5_0_0.land_110.mask(ssp370_tas_cmip6['cesm2'][rea]) # Natural earth masks
    ssp370_tas_eur_cmip6['cesm2'][rea] = ssp370_tas_cmip6['cesm2'][rea].where(mask_eur>15, drop=True).where(mask_land==0, drop=True)

In [ ]:
# hadgem3-gc31-mm

In [ ]:
ssp126_amoc_cmip6['hadgem3-gc31-mm'] = {}
ssp126_amoc_cmip6['hadgem3-gc31-mm']['r1i1p1f3'] = xr.open_dataarray(f"/work/uo1075/m300817/teu_amoc/data/CMIP6/ssp126/hadgem3-gc31-mm/hadgem3-gc31-mm_r1i1p1f3_amoc26_yr.nc", use_cftime=True).to_dataset(name='amoc')

In [ ]:
ssp126_tas_cmip6['hadgem3-gc31-mm'] = {}
ssp126_tas_eur_cmip6['hadgem3-gc31-mm'] = {}
ssp126_tas_cmip6['hadgem3-gc31-mm']['r1i1p1f3'] = xr.open_dataarray(f"/work/uo1075/m300817/teu_amoc/data/CMIP6/ssp126/hadgem3-gc31-mm/hadgem3-gc31-mm_r1i1p1f3_tas_yr.nc", use_cftime=True).to_dataset(name='tas')
mask_eur = regionmask.defined_regions.ar6.land[16, 17, 18, 19].mask(ssp126_tas_cmip6['hadgem3-gc31-mm']['r1i1p1f3'])  # Europe
mask_land = regionmask.defined_regions.natural_earth_v5_0_0.land_110.mask(ssp126_tas_cmip6['hadgem3-gc31-mm']['r1i1p1f3']) # Natural earth masks
ssp126_tas_eur_cmip6['hadgem3-gc31-mm']['r1i1p1f3'] = ssp126_tas_cmip6['hadgem3-gc31-mm']['r1i1p1f3'].where(mask_eur>15, drop=True).where(mask_land==0, drop=True)

In [ ]:
# ec-earth3

In [ ]:
historical_amoc_cmip6['ec-earth3'] = {}
for i in [1,2,4,5,6,7,9,10,12,14,16,17,18,19,20,21,22,23,24,25]:
    rea = f'r{i}i1p1f1'
    historical_amoc_cmip6['ec-earth3'][rea] = xr.open_dataarray(f"/work/uo1075/m300817/teu_amoc/data/CMIP6/historical/ec-earth3/ec-earth3_{rea}_amoc26_yr.nc", use_cftime=True).to_dataset(name='amoc')

In [ ]:
historical_tas_cmip6['ec-earth3'], historical_tas_eur_cmip6['ec-earth3'] = {}, {}
for i in [1,2,4,5,6,7,9,10,12,14,16,17,18,19,20,21,22,23,24,25]:
    rea = f'r{i}i1p1f1'
    historical_tas_cmip6['ec-earth3'][rea] = xr.open_dataarray(f"/work/uo1075/m300817/teu_amoc/data/CMIP6/historical/ec-earth3/ec-earth3_{rea}_tas_yr.nc", use_cftime=True).to_dataset(name='tas')
    mask_eur = regionmask.defined_regions.ar6.land[16, 17, 18, 19].mask(historical_tas_cmip6['ec-earth3'][rea])  # Europe
    mask_land = regionmask.defined_regions.natural_earth_v5_0_0.land_110.mask(historical_tas_cmip6['ec-earth3'][rea]) # Natural earth masks
    historical_tas_eur_cmip6['ec-earth3'][rea] = historical_tas_cmip6['ec-earth3'][rea].where(mask_eur>15, drop=True).where(mask_land==0, drop=True)

In [ ]:
ssp245_amoc_cmip6['ec-earth3'] = {}
for i in [2,7,10,12,14,16,17,18,19,20,21,22,24,25]: #23 also available, not in tas
    rea = f'r{i}i1p1f1'
    ssp245_amoc_cmip6['ec-earth3'][rea] = xr.open_dataarray(f"/work/uo1075/m300817/teu_amoc/data/CMIP6/ssp245/ec-earth3/ec-earth3_{rea}_amoc26_yr.nc", use_cftime=True).to_dataset(name='amoc')

In [ ]:
ssp245_tas_cmip6['ec-earth3'], ssp245_tas_eur_cmip6['ec-earth3'] = {}, {}
for i in [2,7,10,12,14,16,17,18,19,20,21,22,24,25]:
    rea = f'r{i}i1p1f1'
    ssp245_tas_cmip6['ec-earth3'][rea] = xr.open_dataarray(f"/work/uo1075/m300817/teu_amoc/data/CMIP6/ssp245/ec-earth3/ec-earth3_{rea}_tas_yr.nc", use_cftime=True).to_dataset(name='tas')
    mask_eur = regionmask.defined_regions.ar6.land[16, 17, 18, 19].mask(ssp245_tas_cmip6['ec-earth3'][rea])  # Europe
    mask_land = regionmask.defined_regions.natural_earth_v5_0_0.land_110.mask(ssp245_tas_cmip6['ec-earth3'][rea]) # Natural earth masks
    ssp245_tas_eur_cmip6['ec-earth3'][rea] = ssp245_tas_cmip6['ec-earth3'][rea].where(mask_eur>15, drop=True).where(mask_land==0, drop=True)

In [ ]:
# giss-e2-1-g 

In [ ]:
for rea in ['r2i1p1f2', 'r3i1p1f2', 'r4i1p1f2', 'r5i1p1f2']:
    ssp126_amoc_cmip6['giss-e2-1-g'][rea] = xr.open_dataarray(f"/work/uo1075/m300817/teu_amoc/data/CMIP6/ssp126/giss-e2-1-g/giss-e2-1-g_{rea}_amoc26_yr.nc", use_cftime=True).to_dataset(name='amoc')

In [ ]:
for rea in ['r2i1p1f2', 'r3i1p1f2', 'r4i1p1f2', 'r5i1p1f2']:
    ssp126_tas_cmip6['giss-e2-1-g'][rea] = xr.open_dataarray(f"/work/uo1075/m300817/teu_amoc/data/CMIP6/ssp126/giss-e2-1-g/giss-e2-1-g_{rea}_tas_yr.nc", use_cftime=True).to_dataset(name='tas')
    mask_eur = regionmask.defined_regions.ar6.land[16, 17, 18, 19].mask(ssp126_tas_cmip6['giss-e2-1-g'][rea])  # Europe
    mask_land = regionmask.defined_regions.natural_earth_v5_0_0.land_110.mask(ssp126_tas_cmip6['giss-e2-1-g'][rea]) # Natural earth masks
    ssp126_tas_eur_cmip6['giss-e2-1-g'][rea] = ssp126_tas_cmip6['giss-e2-1-g'][rea].where(mask_eur>15, drop=True).where(mask_land==0, drop=True)

In [ ]:
# mri-esm2-0

In [ ]:
for rea in ['r2i1p1f1', 'r3i1p1f1', 'r4i1p1f1', 'r5i1p1f1']:
    ssp126_amoc_cmip6['mri-esm2-0'][rea] = xr.open_dataarray(f"/work/uo1075/m300817/teu_amoc/data/CMIP6/ssp126/mri-esm2-0/mri-esm2-0_{rea}_amoc26_yr.nc", use_cftime=True).to_dataset(name='amoc')

In [ ]:
for rea in ['r2i1p1f1', 'r3i1p1f1', 'r4i1p1f1', 'r5i1p1f1']:
    ssp126_tas_cmip6['mri-esm2-0'][rea] = xr.open_dataarray(f"/work/uo1075/m300817/teu_amoc/data/CMIP6/ssp126/mri-esm2-0/mri-esm2-0_{rea}_tas_yr.nc", use_cftime=True).to_dataset(name='tas')
    mask_eur = regionmask.defined_regions.ar6.land[16, 17, 18, 19].mask(ssp126_tas_cmip6['mri-esm2-0'][rea])  # Europe
    mask_land = regionmask.defined_regions.natural_earth_v5_0_0.land_110.mask(ssp126_tas_cmip6['mri-esm2-0'][rea]) # Natural earth masks
    ssp126_tas_eur_cmip6['mri-esm2-0'][rea] = ssp126_tas_cmip6['mri-esm2-0'][rea].where(mask_eur>15, drop=True).where(mask_land==0, drop=True)

for rea in ['r2i1p1f1', 'r3i1p1f1', 'r4i1p1f1', 'r5i1p1f1']:
    ssp245_tas_cmip6['mri-esm2-0'][rea] = xr.open_dataarray(f"/work/uo1075/m300817/teu_amoc/data/CMIP6/ssp245/mri-esm2-0/mri-esm2-0_{rea}_tas_yr.nc", use_cftime=True).to_dataset(name='tas')
    mask_eur = regionmask.defined_regions.ar6.land[16, 17, 18, 19].mask(ssp245_tas_cmip6['mri-esm2-0'][rea])  # Europe
    mask_land = regionmask.defined_regions.natural_earth_v5_0_0.land_110.mask(ssp245_tas_cmip6['mri-esm2-0'][rea]) # Natural earth masks
    ssp245_tas_eur_cmip6['mri-esm2-0'][rea] = ssp245_tas_cmip6['mri-esm2-0'][rea].where(mask_eur>15, drop=True).where(mask_land==0, drop=True)

NAHosMIP

In [ ]:
nahosmip_lst = ["CanESM5", "CESM2", "EC-Earth3", "HadGEM3-GC3-1LL", "HadGEM3-GC3-1MM",
                "IPSL-CM6A-LR", "MPI-ESM1-2-LR", "MPI-ESM1-2-HR"]

In [ ]:
# AMOC
nahosmip_amoc_yr = {}
for model in nahosmip_lst:
    nahosmip_amoc_yr[model] = xr.open_dataarray(f"/work/uo1075/m300817/hosing/nahosmip/post/{model}_u03-hos_amoc26_yr.nc", 
                                                use_cftime=True).to_dataset(name='amoc').assign_coords(model=model)

In [ ]:
# TAS
nahosmip_tas_yr = {}
for model in nahosmip_lst:
    nahosmip_tas_yr[model] = xr.open_dataarray(f"/work/uo1075/m300817/hosing/nahosmip/post/{model}_u03-hos_tas_yr.nc", 
                                               use_cftime=True).to_dataset(name='tas').assign_coords(model=model)

In [ ]:
# TAS EUR
nahosmip_tas_eur_yr = {}
for model in nahosmip_lst:
    nahosmip_tas_eur_yr[model] = xr.open_dataarray(f"/work/uo1075/m300817/hosing/nahosmip/post/{model}_u03-hos_tas_eur_yr.nc", 
                                               use_cftime=True).to_dataset(name='tas').assign_coords(model=model)

ERA5

In [ ]:
era5_tas_yr = xr.open_mfdataset("/work/uo1075/m300817/teu_amoc/data/ERA5/tas_yr_era_*.nc", use_cftime=True) 

In [ ]:
mask_eur = regionmask.defined_regions.ar6.land[16, 17, 18, 19].mask(era5_tas_yr)  # Europe
mask_land = regionmask.defined_regions.natural_earth_v5_0_0.land_110.mask(era5_tas_yr) # Natural earth masks
era5_tas_eur_yr = era5_tas_yr.where(mask_eur>15, drop=True).where(mask_land==0, drop=True)

# CMIP6

In [ ]:
sce_amoc_cmip6 = {'ssp126':ssp126_amoc_cmip6, 'ssp245':ssp245_amoc_cmip6, 'ssp370':ssp370_amoc_cmip6}
sce_tas_eur_cmip6 = {'ssp126':ssp126_tas_eur_cmip6, 'ssp245':ssp245_tas_eur_cmip6, 'ssp370':ssp370_tas_eur_cmip6}
sce_tas_cmip6 = {'ssp126':ssp126_tas_cmip6, 'ssp245':ssp245_tas_cmip6, 'ssp370':ssp370_tas_cmip6}

In [ ]:
dict_cmip6_marker={
    "access-cm2": ".",
    "access-esm1-5": "+",
    "canesm5": "v",
    "canesm5-canoe": "^",
    "cesm2": "x",
    "cesm2-waccm": "2", 
    'cnrm-cm6-1': "1",
    'cnrm-esm2-1': "3", 
    'ec-earth3': "$o$",
    "fgoals-f3-l": "<",
    "giss-e2-1-g": ">",
    'hadgem3-gc31-ll': "4",
    'hadgem3-gc31-mm': "_",
    "inm-cm4-8": "s",
    "inm-cm5-0": "p",
    'ipsl-cm6a-lr': "8",
    "miroc-es2l": "d",
    "miroc6": "P",
    "mpi-esm1-2-lr": "o",
    "mpi-esm1-2-hr": "*",
    "mri-esm2-0": "h",
    "noresm2-lm": "X", 
    "noresm2-mm": "D",
    'ukesm1-0-ll': "|"
}

In [ ]:
for i, sce in enumerate(["ssp126", "ssp245", "ssp370"]):
    for model in sce_amoc_cmip6[sce]:
        if (model in historical_amoc_cmip6 and model in sce_tas_eur_cmip6[sce] and model in historical_tas_cmip6):
            for rea in sce_amoc_cmip6[sce][model]:
                if (rea in historical_amoc_cmip6[model] and rea in sce_tas_eur_cmip6[sce][model] and rea in historical_tas_cmip6[model]):
                    print(sce+", "+model)
                    break

In [ ]:
fig = plt.figure(figsize=(40, 12))
gs = GridSpec(2, 3) #gs.update(wspace=0.2, hspace=0.5)
for i, sce in enumerate(["ssp126", "ssp245", "ssp370"]):
    countmod, countsim = 0,0
    ax1 = fig.add_subplot(gs[0, 0+i])
    for model in sce_amoc_cmip6[sce]:
        if (model in historical_amoc_cmip6 and model in sce_tas_eur_cmip6[sce] and model in historical_tas_cmip6):
            for rea in sce_amoc_cmip6[sce][model]:
                if (rea in historical_amoc_cmip6[model] and rea in sce_tas_eur_cmip6[sce][model] and rea in historical_tas_cmip6[model]):
                    initial = historical_amoc_cmip6[model][rea].amoc.isel(time=slice(0,50)).mean("time")
                    ax1.plot(sce_amoc_cmip6[sce][model][rea].time.dt.year, 
                             100*(sce_amoc_cmip6[sce][model][rea].rolling(time=10,center=True).mean().amoc-initial)/initial, alpha=0.5, color=ssp_col[sce])
                    break
    initial = his_amoc_yr.isel(realiz=0).amoc.isel(time=slice(0,50)).mean("time")
    ax1.plot(ssp_amoc_yr.isel(realiz=0).sel(scenar=sce).time.dt.year, 
             100*(ssp_amoc_yr.isel(realiz=0).sel(scenar=sce).rolling(time=10,center=True).mean().amoc-initial)/initial,
             alpha=1, color=ssp_col[sce], linestyle='dashed', label='MPI-ESM-LR', linewidth=5)
    ax1.spines.right.set_visible(False)
    ax1.spines.top.set_visible(False)
    ax1.spines.bottom.set_visible(False)
    ax1.spines.left.set_visible(False)
    ax1.set_xlim(2015,2100)
    ax1.set_ylim(-60,20)
    
    ax2 = fig.add_subplot(gs[1, 0+i])
    for model in sce_tas_eur_cmip6[sce].keys():
        if model in historical_tas_cmip6.keys():
            countmod = countmod+1
            for rea in sce_tas_eur_cmip6[sce][model]:
                if (rea in historical_tas_cmip6[model] and rea in sce_amoc_cmip6[sce][model] and rea in historical_amoc_cmip6[model]):
                    countsim = countsim+1
                    initial = weighted_area_lat(historical_tas_eur_cmip6[model][rea]).mean('lat').mean('lon').isel(time=slice(0,50)).mean("time").tas
                    ax2.plot(sce_tas_eur_cmip6[sce][model][rea].time.dt.year,
                             weighted_area_lat(sce_tas_eur_cmip6[sce][model][rea]).mean('lat').mean('lon').rolling(time=10, center=True).mean().tas-initial,
                             alpha=0.5, color=ssp_col[sce])
                    break
            if countsim == 0:
                    countmod = countmod-1
    initial = weighted_area_lat(his_tas_eur_yr.isel(realiz=0)).mean('lat').mean('lon').isel(time=slice(0,50)).mean("time").tas
    ax2.plot(ssp_tas_eur_yr.isel(realiz=0).sel(scenar=sce).time.dt.year, 
             weighted_area_lat(ssp_tas_eur_yr.isel(realiz=0).sel(scenar=sce)).mean('lat').mean('lon').rolling(time=10,center=True).mean().tas-initial, 
             alpha=1, color=ssp_col[sce], linestyle='dashed', label='MPI-ESM-LR', linewidth=5)
    countmod = countmod+1 # add mpi-esm-lr
    ax2.set_xlabel("time [year]")
    ax2.yaxis.tick_right()
    ax2.yaxis.set_label_position("right")
    ax2.spines.left.set_visible(False)
    ax2.spines.top.set_visible(False)
    ax2.spines.right.set_visible(False)
    ax2.spines['left'].set_position(('outward', 10))  # Move the left spine outward by 10 points
    ax2.spines['bottom'].set_position(('outward', 10)) 
    ax2.set_xlim(2015,2100)
    ax2.set_ylim(0, 9)
    ax1.set_title(f"{sce} CMIP6, {countmod} models")

initial = historical_amoc_cmip6["cesm2"]['r10i1p1f1'].amoc.isel(time=slice(0,50)).mean("time")
fig.axes[0].plot(sce_amoc_cmip6["ssp126"]["cesm2"]['r10i1p1f1'].time.dt.year, 
         100*(sce_amoc_cmip6["ssp126"]["cesm2"]['r10i1p1f1'].rolling(time=10,center=True).mean().amoc-initial)/initial, 
         alpha=1,  color=ssp_col["ssp126"], linestyle='dotted', label='CESM2', linewidth=5)
fig.axes[2].plot(sce_amoc_cmip6["ssp245"]["cesm2"]['r10i1p1f1'].time.dt.year, 
         100*(sce_amoc_cmip6["ssp245"]["cesm2"]['r10i1p1f1'].rolling(time=10,center=True).mean().amoc-initial)/initial, 
         alpha=1,  color=ssp_col["ssp245"], linestyle='dotted', label='CESM2', linewidth=5)
fig.axes[4].plot(sce_amoc_cmip6["ssp370"]["cesm2"]['r10i1p1f1'].time.dt.year, 
         100*(sce_amoc_cmip6["ssp370"]["cesm2"]['r10i1p1f1'].rolling(time=10,center=True).mean().amoc-initial)/initial, 
         alpha=1,  color=ssp_col["ssp370"], linestyle='dotted', label='CESM2', linewidth=5)

initial = weighted_area_lat(historical_tas_eur_cmip6["cesm2"]['r10i1p1f1']).mean('lat').mean('lon').isel(time=slice(0,50)).mean("time").tas
fig.axes[1].plot(sce_tas_eur_cmip6["ssp126"]["cesm2"]['r10i1p1f1'].time.dt.year, 
         weighted_area_lat(sce_tas_eur_cmip6["ssp126"]["cesm2"]['r10i1p1f1']).mean('lat').mean('lon').rolling(time=10,center=True).mean().tas-initial, 
         alpha=1, color=ssp_col["ssp126"], linestyle='dotted', label='CESM2', linewidth=5) 
fig.axes[3].plot(sce_tas_eur_cmip6["ssp245"]["cesm2"]['r10i1p1f1'].time.dt.year, 
         weighted_area_lat(sce_tas_eur_cmip6["ssp245"]["cesm2"]['r10i1p1f1']).mean('lat').mean('lon').rolling(time=10,center=True).mean().tas-initial, 
         alpha=1, color=ssp_col["ssp245"], linestyle='dotted', label='CESM2', linewidth=5) 
fig.axes[5].plot(sce_tas_eur_cmip6["ssp370"]["cesm2"]['r10i1p1f1'].time.dt.year, 
         weighted_area_lat(sce_tas_eur_cmip6["ssp370"]["cesm2"]['r10i1p1f1']).mean('lat').mean('lon').rolling(time=10,center=True).mean().tas-initial, 
         alpha=1, color=ssp_col["ssp370"], linestyle='dotted', label='CESM2', linewidth=5) 
fig.axes[0].set_ylabel('AMOC weakening w.r.t. 1850-1899 [%]')
fig.axes[5].set_ylabel(r"$\Delta T_{EU}$ w.r.t. 1850-1899 [°C]")
fig.axes[1].legend(loc='upper center')
fig.axes[3].legend(loc='upper center')
fig.axes[5].legend(loc='upper center')
fig.axes[0].spines.left.set_visible(True)
fig.axes[5].spines.right.set_visible(True)
fig.savefig('../plots/cmip6_amoc&tas_timeseries.pdf', bbox_inches='tight', transparent=True)

In [ ]:
fig = plt.figure(figsize=(40, 12))
gs = GridSpec(2, 3) #gs.update(wspace=0.2, hspace=0.5)
for i, sce in enumerate(["ssp126", "ssp245", "ssp370"]):
    countmod, countsim = 0,0
    ax1 = fig.add_subplot(gs[0, 0+i])
    for model in sce_amoc_cmip6[sce]:
        if (model in historical_amoc_cmip6 and model in sce_tas_eur_cmip6[sce] and model in historical_tas_cmip6):
            for rea in sce_amoc_cmip6[sce][model]:
                if (rea in historical_amoc_cmip6[model] and rea in sce_tas_eur_cmip6[sce][model] and rea in historical_tas_cmip6[model]):
                    initial = historical_amoc_cmip6[model][rea].amoc.isel(time=slice(0,50)).mean("time")
                    ax1.plot(sce_amoc_cmip6[sce][model][rea].time.dt.year, 
                             100*(sce_amoc_cmip6[sce][model][rea].rolling(time=10,center=True).mean().amoc-initial)/initial, alpha=0.3, color=ssp_col[sce])
                    #break
    initial = his_amoc_yr.isel(realiz=0).amoc.isel(time=slice(0,50)).mean("time")
    ax1.plot(ssp_amoc_yr.isel(realiz=0).sel(scenar=sce).time.dt.year, 
             100*(ssp_amoc_yr.isel(realiz=0).sel(scenar=sce).rolling(time=10,center=True).mean().amoc-initial)/initial,
             alpha=1, color=ssp_col[sce], linestyle='dashed', label='MPI-ESM-LR', linewidth=5)
    ax1.spines.right.set_visible(False)
    ax1.spines.top.set_visible(False)
    ax1.spines.bottom.set_visible(False)
    ax1.spines.left.set_visible(False)
    ax1.set_xlim(2015,2100)
    ax1.set_ylim(-60,20)
    
    ax2 = fig.add_subplot(gs[1, 0+i])
    for model in sce_tas_eur_cmip6[sce].keys():
        if model in historical_tas_cmip6.keys():
            countmod = countmod+1
            for rea in sce_tas_eur_cmip6[sce][model]:
                if (rea in historical_tas_cmip6[model] and rea in sce_amoc_cmip6[sce][model] and rea in historical_amoc_cmip6[model]):
                    countsim = countsim+1
                    initial = weighted_area_lat(historical_tas_eur_cmip6[model][rea]).mean('lat').mean('lon').isel(time=slice(0,50)).mean("time").tas
                    ax2.plot(sce_tas_eur_cmip6[sce][model][rea].time.dt.year,
                             weighted_area_lat(sce_tas_eur_cmip6[sce][model][rea]).mean('lat').mean('lon').rolling(time=10, center=True).mean().tas-initial,
                             alpha=0.3, color=ssp_col[sce])
                    #break
            if countsim == 0:
                    countmod = countmod-1
    initial = weighted_area_lat(his_tas_eur_yr.isel(realiz=0)).mean('lat').mean('lon').isel(time=slice(0,50)).mean("time").tas
    ax2.plot(ssp_tas_eur_yr.isel(realiz=0).sel(scenar=sce).time.dt.year, 
             weighted_area_lat(ssp_tas_eur_yr.isel(realiz=0).sel(scenar=sce)).mean('lat').mean('lon').rolling(time=10,center=True).mean().tas-initial, 
             alpha=1, color=ssp_col[sce], linestyle='dashed', label='MPI-ESM-LR', linewidth=5)
    countmod = countmod+1 # add mpi-esm-lr
    ax2.set_xlabel("time [year]")
    ax2.yaxis.tick_right()
    ax2.yaxis.set_label_position("right")
    ax2.spines.left.set_visible(False)
    ax2.spines.top.set_visible(False)
    ax2.spines.right.set_visible(False)
    ax2.spines['left'].set_position(('outward', 10))  # Move the left spine outward by 10 points
    ax2.spines['bottom'].set_position(('outward', 10)) 
    ax2.set_xlim(2015,2100)
    ax2.set_ylim(0, 9)
    ax1.set_title(f"{sce} CMIP6, {countsim} simulations from {countmod} models")

initial = historical_amoc_cmip6["cesm2"]['r10i1p1f1'].amoc.isel(time=slice(0,50)).mean("time")
fig.axes[0].plot(sce_amoc_cmip6["ssp126"]["cesm2"]['r10i1p1f1'].time.dt.year, 
         100*(sce_amoc_cmip6["ssp126"]["cesm2"]['r10i1p1f1'].rolling(time=10,center=True).mean().amoc-initial)/initial, 
         alpha=1,  color=ssp_col["ssp126"], linestyle='dotted', label='CESM2', linewidth=5)
fig.axes[2].plot(sce_amoc_cmip6["ssp245"]["cesm2"]['r10i1p1f1'].time.dt.year, 
         100*(sce_amoc_cmip6["ssp245"]["cesm2"]['r10i1p1f1'].rolling(time=10,center=True).mean().amoc-initial)/initial, 
         alpha=1,  color=ssp_col["ssp245"], linestyle='dotted', label='CESM2', linewidth=5)
fig.axes[4].plot(sce_amoc_cmip6["ssp370"]["cesm2"]['r10i1p1f1'].time.dt.year, 
         100*(sce_amoc_cmip6["ssp370"]["cesm2"]['r10i1p1f1'].rolling(time=10,center=True).mean().amoc-initial)/initial, 
         alpha=1,  color=ssp_col["ssp370"], linestyle='dotted', label='CESM2', linewidth=5)

initial = weighted_area_lat(historical_tas_eur_cmip6["cesm2"]['r10i1p1f1']).mean('lat').mean('lon').isel(time=slice(0,50)).mean("time").tas
fig.axes[1].plot(sce_tas_eur_cmip6["ssp126"]["cesm2"]['r10i1p1f1'].time.dt.year, 
         weighted_area_lat(sce_tas_eur_cmip6["ssp126"]["cesm2"]['r10i1p1f1']).mean('lat').mean('lon').rolling(time=10,center=True).mean().tas-initial, 
         alpha=1, color=ssp_col["ssp126"], linestyle='dotted', label='CESM2', linewidth=5) 
fig.axes[3].plot(sce_tas_eur_cmip6["ssp245"]["cesm2"]['r10i1p1f1'].time.dt.year, 
         weighted_area_lat(sce_tas_eur_cmip6["ssp245"]["cesm2"]['r10i1p1f1']).mean('lat').mean('lon').rolling(time=10,center=True).mean().tas-initial, 
         alpha=1, color=ssp_col["ssp245"], linestyle='dotted', label='CESM2', linewidth=5) 
fig.axes[5].plot(sce_tas_eur_cmip6["ssp370"]["cesm2"]['r10i1p1f1'].time.dt.year, 
         weighted_area_lat(sce_tas_eur_cmip6["ssp370"]["cesm2"]['r10i1p1f1']).mean('lat').mean('lon').rolling(time=10,center=True).mean().tas-initial, 
         alpha=1, color=ssp_col["ssp370"], linestyle='dotted', label='CESM2', linewidth=5) 
fig.axes[0].set_ylabel('AMOC weakening w.r.t. 1850-1899 [%]')
fig.axes[5].set_ylabel(r"$\Delta T_{EU}$ w.r.t. 1850-1899 [°C]")
fig.axes[1].legend(loc='upper center')
fig.axes[3].legend(loc='upper center')
fig.axes[5].legend(loc='upper center')
fig.axes[0].spines.left.set_visible(True)
fig.axes[5].spines.right.set_visible(True)
#fig.savefig('../plots/cmip6_amoc&tas_timeseries.pdf', bbox_inches='tight', transparent=True)

In [ ]:
fig = plt.figure(figsize=(40, 12))
gs = GridSpec(2, 3) #gs.update(wspace=0.2, hspace=0.5)
for i, sce in enumerate(["ssp126", "ssp245", "ssp370"]):
    countmod, countsim = 0,0
    ax1 = fig.add_subplot(gs[0, 0+i])
    for model in sce_amoc_cmip6[sce]:
        if model in historical_amoc_cmip6.keys():
            for rea in sce_amoc_cmip6[sce][model]:
                if rea in historical_amoc_cmip6[model].keys():
                    initial = historical_amoc_cmip6[model][rea].amoc.isel(time=slice(0,50)).mean("time")
                    ax1.plot(sce_amoc_cmip6[sce][model][rea].time.dt.year, 
                             100*(sce_amoc_cmip6[sce][model][rea].rolling(time=10,center=True).mean().amoc-initial)/initial, alpha=0.5, color=ssp_col[sce])
                    break
    initial = his_amoc_yr.isel(realiz=0).amoc.isel(time=slice(0,50)).mean("time")
    ax1.plot(ssp_amoc_yr.isel(realiz=0).sel(scenar=sce).time.dt.year, 
             100*(ssp_amoc_yr.isel(realiz=0).sel(scenar=sce).rolling(time=10,center=True).mean().amoc-initial)/initial,
             alpha=1, color=ssp_col[sce], linestyle='dashed', label='MPI-ESM-LR', linewidth=4)
    ax1.spines.right.set_visible(False)
    ax1.spines.top.set_visible(False)
    ax1.spines.bottom.set_visible(False)
    ax1.spines.left.set_visible(False)
    ax1.set_xlim(2020,2100)
    ax1.set_ylim(-60,20)
    
    ax2 = fig.add_subplot(gs[1, 0+i])
    for model in sce_tas_cmip6[sce].keys():
        if model in historical_tas_cmip6.keys():
            countmod = countmod+1
            for rea in sce_tas_cmip6[sce][model]:
                if rea in historical_tas_cmip6[model].keys():
                    countsim = countsim+1
                    initial = weighted_area_lat(historical_tas_cmip6[model][rea]).mean('lat').mean('lon').isel(time=slice(0,50)).mean("time").tas
                    ax2.plot(sce_tas_cmip6[sce][model][rea].time.dt.year,
                             weighted_area_lat(sce_tas_cmip6[sce][model][rea]).mean('lat').mean('lon').rolling(time=10, center=True).mean().tas-initial,
                             alpha=0.5, color=ssp_col[sce])
                    break
            if countsim == 0:
                    countmod = countmod-1
    initial = weighted_area_lat(his_tas_yr.isel(realiz=0)).mean('lat').mean('lon').isel(time=slice(0,50)).mean("time").tas
    ax2.plot(ssp_tas_yr.isel(realiz=0).sel(scenar=sce).time.dt.year, 
             weighted_area_lat(ssp_tas_yr.isel(realiz=0).sel(scenar=sce)).mean('lat').mean('lon').rolling(time=10,center=True).mean().tas-initial, 
             alpha=1, color=ssp_col[sce], linestyle='dashed', label='MPI-ESM-LR', linewidth=4)
    countmod = countmod+1 # add mpi-esm-lr
    ax2.set_xlabel("time [year]")
    ax2.yaxis.tick_right()
    ax2.yaxis.set_label_position("right")
    ax2.spines.left.set_visible(False)
    ax2.spines.top.set_visible(False)
    ax2.spines.right.set_visible(False)
    ax2.spines['left'].set_position(('outward', 10))  # Move the left spine outward by 10 points
    ax2.spines['bottom'].set_position(('outward', 10)) 
    ax2.set_xlim(2020,2100)
    ax2.set_ylim(0, 7)
    ax1.set_title(f"{sce} CMIP6, {countmod} models")

initial = historical_amoc_cmip6["cesm2"]['r10i1p1f1'].amoc.isel(time=slice(0,50)).mean("time")
fig.axes[0].plot(sce_amoc_cmip6["ssp126"]["cesm2"]['r10i1p1f1'].time.dt.year, 
         100*(sce_amoc_cmip6["ssp126"]["cesm2"]['r10i1p1f1'].rolling(time=10,center=True).mean().amoc-initial)/initial, 
         alpha=1,  color=ssp_col["ssp126"], linestyle='dotted', label='CESM2', linewidth=4)
fig.axes[2].plot(sce_amoc_cmip6["ssp245"]["cesm2"]['r10i1p1f1'].time.dt.year, 
         100*(sce_amoc_cmip6["ssp245"]["cesm2"]['r10i1p1f1'].rolling(time=10,center=True).mean().amoc-initial)/initial, 
         alpha=1,  color=ssp_col["ssp245"], linestyle='dotted', label='CESM2', linewidth=4)
fig.axes[4].plot(sce_amoc_cmip6["ssp370"]["cesm2"]['r10i1p1f1'].time.dt.year, 
         100*(sce_amoc_cmip6["ssp370"]["cesm2"]['r10i1p1f1'].rolling(time=10,center=True).mean().amoc-initial)/initial, 
         alpha=1,  color=ssp_col["ssp370"], linestyle='dotted', label='CESM2', linewidth=4)

initial = weighted_area_lat(historical_tas_cmip6["cesm2"]['r10i1p1f1']).mean('lat').mean('lon').isel(time=slice(0,50)).mean("time").tas
fig.axes[1].plot(sce_tas_cmip6["ssp126"]["cesm2"]['r10i1p1f1'].time.dt.year, 
         weighted_area_lat(sce_tas_cmip6["ssp126"]["cesm2"]['r10i1p1f1']).mean('lat').mean('lon').rolling(time=10,center=True).mean().tas-initial, 
         alpha=1, color=ssp_col["ssp126"], linestyle='dotted', label='CESM2', linewidth=4) 
fig.axes[3].plot(sce_tas_cmip6["ssp245"]["cesm2"]['r10i1p1f1'].time.dt.year, 
         weighted_area_lat(sce_tas_cmip6["ssp245"]["cesm2"]['r10i1p1f1']).mean('lat').mean('lon').rolling(time=10,center=True).mean().tas-initial, 
         alpha=1, color=ssp_col["ssp245"], linestyle='dotted', label='CESM2', linewidth=4) 
fig.axes[5].plot(sce_tas_cmip6["ssp370"]["cesm2"]['r10i1p1f1'].time.dt.year, 
         weighted_area_lat(sce_tas_cmip6["ssp370"]["cesm2"]['r10i1p1f1']).mean('lat').mean('lon').rolling(time=10,center=True).mean().tas-initial, 
         alpha=1, color=ssp_col["ssp370"], linestyle='dotted', label='CESM2', linewidth=4) 

fig.axes[0].set_ylabel('AMOC weakening w.r.t. 1850-1899 [%]')
fig.axes[5].set_ylabel(r"$\Delta T$ w.r.t. 1850-1899 [°C]")
fig.axes[1].legend(loc='upper center')
fig.axes[3].legend(loc='upper center')
fig.axes[5].legend(loc='upper center')
fig.axes[0].spines.left.set_visible(True)
fig.axes[5].spines.right.set_visible(True)
fig.savefig('../plots/cmip6_amoc&tas_timeseries_glob.png', bbox_inches='tight', transparent=True)

In [ ]:
fig = plt.figure(figsize=(10, 10))

for model in historical_tas_cmip6:
    if model in dict_cmip6_marker:
        amoc_refs = [historical_amoc_cmip6[model][rea].amoc.isel(time=slice(0, 50)).mean("time").item()
                     for rea in historical_amoc_cmip6[model]]
        tas_refs = [weighted_area_lat(historical_tas_eur_cmip6[model][rea]).mean("lat").mean("lon").tas.isel(time=slice(0, 50)).mean("time").item() - 273.15
                    for rea in historical_tas_cmip6[model]]
        plt.scatter(sum(amoc_refs) / len(amoc_refs), sum(tas_refs) / len(tas_refs), color="black", marker=dict_cmip6_marker[model], s=70)
        
amoc_ref = his_amoc_yr.amoc.mean("realiz").isel(time=slice(0,50)).mean("time")
tas_eur_ref = weighted_area_lat(his_tas_eur_yr.mean("realiz")).mean('lat').mean('lon').tas.isel(time=slice(0,50)).mean("time")
plt.scatter((amoc_ref), (tas_eur_ref-273.15), color='black', marker = dict_cmip6_marker["mpi-esm1-2-lr"], s=70)

plt.xlabel(r"AMOC strength in 1850-1899 [Sv]")
plt.ylabel(r"$\mathrm{T}_{\mathrm{EU}}$ in 1850-1899 [°C]")
legend_handles = [Line2D([0], [0], marker=marker, color=(0, 0, 0, 0), label=model, 
                         markerfacecolor='darkgrey', markeredgecolor='darkgrey', markersize=8) 
                  for model, marker in dict_cmip6_marker.items()]
plt.legend(handles=legend_handles, bbox_to_anchor=(1.05, 1), frameon=False)
ax = plt.gca()
ax.spines['right'].set_visible(False)
ax.spines['top'].set_visible(False)
ax.spines['left'].set_position(('outward', 10))  # Move the left spine outward by 10 points
ax.spines['bottom'].set_position(('outward', 10)) 
fig.savefig('../plots/cmip6_amoc&tas_scatter_pi.pdf', bbox_inches='tight', transparent=True)

In [ ]:
his_amoc_yr.amoc.mean("realiz").isel(time=slice(-10,None)).mean("time")

In [ ]:
his_amoc_yr.amoc.std("realiz").isel(time=slice(-10,None)).mean("time")

In [ ]:
#################################### Load RAPID #####################################
outpath = "/work/uo1075/m300817/thesis/data/RAPID/"

rapid = xr.load_dataset(outpath+"moc_transports_yearmean_v2024.nc", use_cftime=True)
rapid_amoc_yearmean = rapid['moc_mar_hc10']
time_rapid_yearmean = rapid_amoc_yearmean.time.dt.year.values

In [ ]:
rapid_amoc_yearmean.isel(time=slice(1,11)).mean("time")

In [ ]:
fig = plt.figure(figsize=(10, 10))

for i, sce in enumerate(["ssp126", "ssp245", "ssp370"]):
    for model in sce_tas_eur_cmip6[sce]:
        if (model in historical_amoc_cmip6 and model in sce_tas_eur_cmip6[sce] and model in historical_tas_cmip6):
            for rea in sce_tas_eur_cmip6[sce][model]:
                if (rea in historical_amoc_cmip6[model] and rea in sce_tas_eur_cmip6[sce][model] and rea in historical_tas_cmip6[model]):
                    amoc_ref = historical_amoc_cmip6[model][rea].amoc.isel(time=slice(0,50)).mean("time")
                    amoc_2100 = sce_amoc_cmip6[sce][model][rea].amoc.isel(time=slice(-10,None)).mean("time")
                    tas_eur_ref = weighted_area_lat(historical_tas_eur_cmip6[model][rea]).mean('lat').mean('lon').tas.isel(time=slice(0,50)).mean("time")
                    tas_eur_2100 = weighted_area_lat(sce_tas_eur_cmip6[sce][model][rea]).mean('lat').mean('lon').tas.isel(time=slice(-10,None)).mean("time")
                    plt.scatter(amoc_2100, (tas_eur_2100)-273.15, color=ssp_col[sce], marker  = dict_cmip6_marker[model], s=70)
                    break
    amoc_ref = his_amoc_yr.amoc.isel(realiz=0).isel(time=slice(0,50)).mean("time")
    amoc_2100 = ssp_amoc_yr.sel(scenar=sce).isel(realiz=0).amoc.isel(time=slice(-10,None)).mean("time")
    tas_eur_ref = weighted_area_lat(his_tas_eur_yr.isel(realiz=0)).mean('lat').mean('lon').tas.isel(time=slice(0,50)).mean("time")
    tas_eur_2100 = weighted_area_lat(ssp_tas_eur_yr.sel(scenar=sce).isel(realiz=0)).mean('lat').mean('lon').tas.isel(time=slice(-10,None)).mean("time")
    plt.scatter((amoc_2100), (tas_eur_2100-273.15), color=ssp_col[sce], marker = dict_cmip6_marker["mpi-esm1-2-lr"], s=70)
    
plt.xlabel(r"AMOC strength in 2091-2100 [Sv]")
plt.ylabel(r"$\mathrm{T}_{\mathrm{EU}}$ in 2091-2100 [°C]")
legend_handles = [Line2D([0], [0], marker=marker, color=(0, 0, 0, 0), label=model, 
                         markerfacecolor='darkgrey', markeredgecolor='darkgrey', markersize=8) 
                  for model, marker in dict_cmip6_marker.items()]
plt.legend(handles=legend_handles, bbox_to_anchor=(1.05, 1), frameon=False)
ax = plt.gca()
ax.spines['right'].set_visible(False)
ax.spines['top'].set_visible(False)
ax.spines['left'].set_position(('outward', 10))  # Move the left spine outward by 10 points
ax.spines['bottom'].set_position(('outward', 10)) 
#fig.savefig('../plots/cmip6_amocabs&tasabs_scatter_2100.pdf', bbox_inches='tight', transparent=True)

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(9, 20), sharex=True, constrained_layout=False)
for i, sce in enumerate(["ssp126", "ssp245", "ssp370"]):
    for model in sce_tas_eur_cmip6[sce]:
        if (model in historical_amoc_cmip6 and model in sce_tas_eur_cmip6[sce] and model in historical_tas_cmip6):
            amoc_ref, tas_eur_ref = [], []
            amoc_2100, tas_eur_2100 = [], []
            for rea in sce_tas_eur_cmip6[sce][model]:
                if (rea in historical_amoc_cmip6[model] and rea in sce_tas_eur_cmip6[sce][model] and rea in historical_tas_cmip6[model]):
                    amoc_ref.append(historical_amoc_cmip6[model][rea].amoc.isel(time=slice(0,50)).mean("time"))
                    amoc_2100.append(sce_amoc_cmip6[sce][model][rea].amoc.isel(time=slice(-10,None)).mean("time"))
                    tas_eur_ref.append(weighted_area_lat(historical_tas_eur_cmip6[model][rea]).mean('lat').mean('lon').tas.isel(time=slice(0,50)).mean("time"))
                    tas_eur_2100.append(weighted_area_lat(sce_tas_eur_cmip6[sce][model][rea]).mean('lat').mean('lon').tas.isel(time=slice(-10,None)).mean("time"))
            ax1.scatter(np.mean(amoc_2100), np.mean(tas_eur_2100)-273.15, color=ssp_col[sce], marker  = dict_cmip6_marker[model], s=70)
            ax2.scatter(np.mean(amoc_2100), np.mean(tas_eur_2100)-np.mean(tas_eur_ref), color=ssp_col[sce], marker  = dict_cmip6_marker[model], s=70)
    
    amoc_ref = his_amoc_yr.amoc.mean("realiz").isel(time=slice(0,50)).mean("time")
    amoc_2100 = ssp_amoc_yr.sel(scenar=sce).mean("realiz").amoc.isel(time=slice(-10,None)).mean("time")
    tas_eur_ref = weighted_area_lat(his_tas_eur_yr.mean("realiz")).mean('lat').mean('lon').tas.isel(time=slice(0,50)).mean("time")
    tas_eur_2100 = weighted_area_lat(ssp_tas_eur_yr.sel(scenar=sce).mean("realiz")).mean('lat').mean('lon').tas.isel(time=slice(-10,None)).mean("time")
    ax1.scatter((amoc_2100), (tas_eur_2100-273.15), color=ssp_col[sce], marker = dict_cmip6_marker["mpi-esm1-2-lr"], s=70)
    ax2.scatter((amoc_2100), (tas_eur_2100-tas_eur_ref), color=ssp_col[sce], marker = dict_cmip6_marker["mpi-esm1-2-lr"], s=70)
    
ax1.set_ylabel(r"$\mathrm{T}_{\mathrm{EU}}$ in 2091-2100 [°C]")
ax2.set_ylabel(r"$\Delta \mathrm{T}_{\mathrm{EU}}$ in 2091-2100 w.r.t. 1850-1899 [°C]")
ax2.set_xlabel(r"AMOC strength in 2091-2100 [Sv]")
legend_handles = [Line2D([0], [0], marker=marker, color=(0, 0, 0, 0), label=model, 
                         markerfacecolor='darkgrey', markeredgecolor='darkgrey', markersize=8) 
                  for model, marker in dict_cmip6_marker.items()]
fig.legend(handles=legend_handles,loc="center left", bbox_to_anchor=(0.83, 0.5), frameon=False)
ax1.spines["top"].set_visible(False)
ax1.spines["right"].set_visible(False)
ax1.spines["bottom"].set_visible(False)
ax1.tick_params(axis="x", which="both", bottom=True, labelbottom=True)
ax2.spines["top"].set_visible(False)
ax2.spines["right"].set_visible(False)
fig.subplots_adjust(right=0.8, hspace=0.15)
fig.savefig('../plots/cmip6_amocabs&tas_scatter_2100.pdf', bbox_inches='tight', transparent=True)

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(9, 20), sharex=True, constrained_layout=False)
for i, sce in enumerate(["ssp126", "ssp245", "ssp370"]):
    for model in sce_tas_eur_cmip6[sce]:
        if (model in historical_amoc_cmip6 and model in sce_tas_eur_cmip6[sce] and model in historical_tas_cmip6):
            amoc_ref, tas_eur_ref = [], []
            amoc_2100, tas_eur_2100 = [], []
            for rea in sce_tas_eur_cmip6[sce][model]:
                if (rea in historical_amoc_cmip6[model] and rea in sce_tas_eur_cmip6[sce][model] and rea in historical_tas_cmip6[model]):
                    amoc_ref.append(historical_amoc_cmip6[model][rea].amoc.isel(time=slice(0,50)).mean("time"))
                    amoc_2100.append(sce_amoc_cmip6[sce][model][rea].amoc.isel(time=slice(-10,None)).mean("time"))
                    tas_eur_ref.append(weighted_area_lat(historical_tas_eur_cmip6[model][rea]).mean('lat').mean('lon').tas.isel(time=slice(0,50)).mean("time"))
                    tas_eur_2100.append(weighted_area_lat(sce_tas_eur_cmip6[sce][model][rea]).mean('lat').mean('lon').tas.isel(time=slice(-10,None)).mean("time"))
            ax1.scatter(100*(np.mean(amoc_2100)-np.mean(amoc_ref))/np.mean(amoc_ref), np.mean(tas_eur_2100)-273.15, color=ssp_col[sce], marker  = dict_cmip6_marker[model], s=70)
            ax2.scatter(100*(np.mean(amoc_2100)-np.mean(amoc_ref))/np.mean(amoc_ref), np.mean(tas_eur_2100)-np.mean(tas_eur_ref), color=ssp_col[sce], marker  = dict_cmip6_marker[model], s=70)
    
    amoc_ref = his_amoc_yr.amoc.mean("realiz").isel(time=slice(0,50)).mean("time")
    amoc_2100 = ssp_amoc_yr.sel(scenar=sce).mean("realiz").amoc.isel(time=slice(-10,None)).mean("time")
    tas_eur_ref = weighted_area_lat(his_tas_eur_yr.mean("realiz")).mean('lat').mean('lon').tas.isel(time=slice(0,50)).mean("time")
    tas_eur_2100 = weighted_area_lat(ssp_tas_eur_yr.sel(scenar=sce).mean("realiz")).mean('lat').mean('lon').tas.isel(time=slice(-10,None)).mean("time")
    ax1.scatter(100*(amoc_2100-amoc_ref)/amoc_ref, (tas_eur_2100-273.15), color=ssp_col[sce], marker = dict_cmip6_marker["mpi-esm1-2-lr"], s=70)
    ax2.scatter(100*(amoc_2100-amoc_ref)/amoc_ref, (tas_eur_2100-tas_eur_ref), color=ssp_col[sce], marker = dict_cmip6_marker["mpi-esm1-2-lr"], s=70)
    
ax1.set_ylabel(r"$\mathrm{T}_{\mathrm{EU}}$ in 2091-2100 [°C]")
ax2.set_ylabel(r"$\Delta \mathrm{T}_{\mathrm{EU}}$ in 2091-2100 w.r.t. 1850-1899 [°C]")
ax2.set_xlabel(r"$\Delta$AMOC strength in 2091-2100 w.r.t. 1850-1899 [%]")

legend_handles = [Line2D([0], [0], marker=marker, color=(0, 0, 0, 0), label=model, 
                         markerfacecolor='darkgrey', markeredgecolor='darkgrey', markersize=8) 
                  for model, marker in dict_cmip6_marker.items()]
fig.legend(handles=legend_handles,loc="center left", bbox_to_anchor=(0.83, 0.5), frameon=False)
ax1.spines["top"].set_visible(False)
ax1.spines["right"].set_visible(False)
ax1.spines["bottom"].set_visible(False)
ax1.tick_params(axis="x", which="both", bottom=True, labelbottom=True)
ax2.spines["top"].set_visible(False)
ax2.spines["right"].set_visible(False)
fig.subplots_adjust(right=0.8, hspace=0.15)
fig.savefig('../plots/cmip6_amoc&tas_changes_scatter_2100.pdf', bbox_inches='tight', transparent=True)

In [ ]:
fig, ((ax1, ax3), (ax2, ax4)) = plt.subplots(2, 2, figsize=(20, 20), sharex='col', constrained_layout=False)
for i, sce in enumerate(["ssp126", "ssp245", "ssp370"]):
    for model in sce_tas_eur_cmip6[sce]:
        if (model in historical_amoc_cmip6 and model in sce_tas_eur_cmip6[sce] and model in historical_tas_cmip6):
            amoc_ref, tas_eur_ref = [], []
            amoc_2100, tas_eur_2100 = [], []
            for rea in sce_tas_eur_cmip6[sce][model]:
                if (rea in historical_amoc_cmip6[model] and rea in sce_tas_eur_cmip6[sce][model] and rea in historical_tas_cmip6[model]):
                    amoc_ref.append(historical_amoc_cmip6[model][rea].amoc.isel(time=slice(0,50)).mean("time"))
                    amoc_2100.append(sce_amoc_cmip6[sce][model][rea].amoc.isel(time=slice(-10,None)).mean("time"))
                    tas_eur_ref.append(weighted_area_lat(historical_tas_eur_cmip6[model][rea]).mean('lat').mean('lon').tas.isel(time=slice(0,50)).mean("time"))
                    tas_eur_2100.append(weighted_area_lat(sce_tas_eur_cmip6[sce][model][rea]).mean('lat').mean('lon').tas.isel(time=slice(-10,None)).mean("time"))
            ax1.scatter(np.mean(amoc_2100), np.mean(tas_eur_2100)-273.15, color=ssp_col[sce], marker  = dict_cmip6_marker[model], s=70)
            ax2.scatter(np.mean(amoc_2100), np.mean(tas_eur_2100)-np.mean(tas_eur_ref), color=ssp_col[sce], marker  = dict_cmip6_marker[model], s=70)
    
            ax3.scatter(100*(np.mean(amoc_2100)-np.mean(amoc_ref))/np.mean(amoc_ref), np.mean(tas_eur_2100)-273.15, color=ssp_col[sce], marker  = dict_cmip6_marker[model], s=70)
            ax4.scatter(100*(np.mean(amoc_2100)-np.mean(amoc_ref))/np.mean(amoc_ref), np.mean(tas_eur_2100)-np.mean(tas_eur_ref), color=ssp_col[sce], marker  = dict_cmip6_marker[model], s=70)
    
    amoc_ref = his_amoc_yr.amoc.mean("realiz").isel(time=slice(0,50)).mean("time")
    amoc_2100 = ssp_amoc_yr.sel(scenar=sce).mean("realiz").amoc.isel(time=slice(-10,None)).mean("time")
    tas_eur_ref = weighted_area_lat(his_tas_eur_yr.mean("realiz")).mean('lat').mean('lon').tas.isel(time=slice(0,50)).mean("time")
    tas_eur_2100 = weighted_area_lat(ssp_tas_eur_yr.sel(scenar=sce).mean("realiz")).mean('lat').mean('lon').tas.isel(time=slice(-10,None)).mean("time")
    ax1.scatter((amoc_2100), (tas_eur_2100-273.15), color=ssp_col[sce], marker = dict_cmip6_marker["mpi-esm1-2-lr"], s=70)
    ax2.scatter((amoc_2100), (tas_eur_2100-tas_eur_ref), color=ssp_col[sce], marker = dict_cmip6_marker["mpi-esm1-2-lr"], s=70)
    
    ax3.scatter(100*(amoc_2100-amoc_ref)/amoc_ref, (tas_eur_2100-273.15), color=ssp_col[sce], marker = dict_cmip6_marker["mpi-esm1-2-lr"], s=70)
    ax4.scatter(100*(amoc_2100-amoc_ref)/amoc_ref, (tas_eur_2100-tas_eur_ref), color=ssp_col[sce], marker = dict_cmip6_marker["mpi-esm1-2-lr"], s=70)
    


ax1.set_ylabel(r"$\mathrm{T}_{\mathrm{EU}}$ in 2091-2100 [°C]")
ax2.set_ylabel(r"$\Delta \mathrm{T}_{\mathrm{EU}}$ in 2091-2100 w.r.t. 1850-1899 [°C]")
ax4.set_xlabel(r"$\Delta$AMOC strength in 2091-2100 w.r.t. 1850-1899 [%]")
ax2.set_xlabel(r"AMOC strength in 2091-2100 [Sv]")


legend_handles = [Line2D([0], [0], marker=marker, color=(0, 0, 0, 0), label=model, 
                         markerfacecolor='darkgrey', markeredgecolor='darkgrey', markersize=8) 
                  for model, marker in dict_cmip6_marker.items()]
fig.legend(handles=legend_handles,loc="center left", bbox_to_anchor=(0.83, 0.5), frameon=False)
ax1.spines["top"].set_visible(False)
ax1.spines["right"].set_visible(False)
ax1.spines["bottom"].set_visible(False)
ax1.tick_params(axis="x", which="both", bottom=True, labelbottom=True)
ax2.spines["top"].set_visible(False)
ax2.spines["right"].set_visible(False)

ax3.spines["top"].set_visible(False)
ax3.spines["right"].set_visible(False)
ax3.spines["bottom"].set_visible(False)
ax3.spines["left"].set_visible(False)
ax3.tick_params(axis="x", which="both", bottom=True, labelbottom=True)
ax4.spines["top"].set_visible(False)
ax4.spines["right"].set_visible(False)
ax4.spines["left"].set_visible(False)
labels = ["a)", "b)", "c)", "d)"]
for ax, lab in zip([ax1, ax2, ax3, ax4], labels):
    ax.text( -0.1, 1.02, lab,transform=ax.transAxes,va="top", ha="left", fontsize=20)
fig.subplots_adjust(right=0.8, hspace=0.15)
fig.savefig('../plots/cmip6_amoc&tas_abs&changes_scatter_2100.pdf', bbox_inches='tight', transparent=True)

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(9, 20), sharex=True, constrained_layout=False)
for i, sce in enumerate(["ssp126", "ssp245", "ssp370"]):
    for model in sce_tas_eur_cmip6[sce]:
        if (model in historical_amoc_cmip6 and model in sce_tas_eur_cmip6[sce] and model in historical_tas_cmip6):
            amoc_ref, tas_eur_ref, tas_glo_ref = [], [], []
            amoc_2100, tas_eur_2100, tas_glo_2100 = [], [], []
            for rea in sce_tas_eur_cmip6[sce][model]:
                if (rea in historical_amoc_cmip6[model] and rea in sce_tas_eur_cmip6[sce][model] and rea in historical_tas_cmip6[model]):
                    amoc_ref.append(historical_amoc_cmip6[model][rea].amoc.isel(time=slice(0,50)).mean("time"))
                    amoc_2100.append(sce_amoc_cmip6[sce][model][rea].amoc.isel(time=slice(-10,None)).mean("time"))
                    tas_eur_ref.append(weighted_area_lat(historical_tas_eur_cmip6[model][rea]).mean('lat').mean('lon').tas.isel(time=slice(0,50)).mean("time"))
                    tas_eur_2100.append(weighted_area_lat(sce_tas_eur_cmip6[sce][model][rea]).mean('lat').mean('lon').tas.isel(time=slice(-10,None)).mean("time"))
                    tas_glo_ref.append(weighted_area_lat(historical_tas_cmip6[model][rea]).mean('lat').mean('lon').tas.isel(time=slice(0,50)).mean("time"))
                    tas_glo_2100.append(weighted_area_lat(sce_tas_cmip6[sce][model][rea]).mean('lat').mean('lon').tas.isel(time=slice(-10,None)).mean("time"))
            
            ax1.scatter(100*(np.mean(amoc_2100)-np.mean(amoc_ref))/np.mean(amoc_ref), 
                        np.mean(tas_glo_2100)-np.mean(tas_glo_ref), 
                        color=ssp_col[sce], marker  = dict_cmip6_marker[model], s=70)
            ax2.scatter(100*(np.mean(amoc_2100)-np.mean(amoc_ref))/np.mean(amoc_ref), 
                        (np.mean(tas_eur_2100)-np.mean(tas_eur_ref))/(np.mean(tas_glo_2100)-np.mean(tas_glo_ref)),
                        color=ssp_col[sce], marker  = dict_cmip6_marker[model], s=70)
    
    amoc_ref = his_amoc_yr.amoc.mean("realiz").isel(time=slice(0,50)).mean("time")
    amoc_2100 = ssp_amoc_yr.sel(scenar=sce).mean("realiz").amoc.isel(time=slice(-10,None)).mean("time")
    tas_eur_ref = weighted_area_lat(his_tas_eur_yr.mean("realiz")).mean('lat').mean('lon').tas.isel(time=slice(0,50)).mean("time")
    tas_eur_2100 = weighted_area_lat(ssp_tas_eur_yr.sel(scenar=sce).mean("realiz")).mean('lat').mean('lon').tas.isel(time=slice(-10,None)).mean("time")
    tas_glo_ref = weighted_area_lat(his_tas_yr.mean("realiz")).mean('lat').mean('lon').tas.isel(time=slice(0,50)).mean("time")
    tas_glo_2100 = weighted_area_lat(ssp_tas_yr.sel(scenar=sce).mean("realiz")).mean('lat').mean('lon').tas.isel(time=slice(-10,None)).mean("time")
 
    ax1.scatter(100*(amoc_2100-amoc_ref)/amoc_ref, tas_glo_2100-tas_glo_ref, 
                color=ssp_col[sce], marker = dict_cmip6_marker["mpi-esm1-2-lr"], s=70)
    ax2.scatter(100*(amoc_2100-amoc_ref)/amoc_ref, (tas_eur_2100-tas_eur_ref)/(tas_glo_2100-tas_glo_ref),
                color=ssp_col[sce], marker = dict_cmip6_marker["mpi-esm1-2-lr"], s=70)
    
ax1.set_ylabel(r"$\mathrm{T}_{\mathrm{GLOB}}$ in 2091-2100 [°C]")
ax2.set_ylabel(r"$\Delta \mathrm{T}_{\mathrm{EU}}$/$\Delta \mathrm{T}_{\mathrm{GLOB}}$ in 2091-2100 w.r.t. 1850-1899 [°C]")

ax2.set_xlabel(r"$\Delta$AMOC strength in 2091-2100 w.r.t. 1850-1899 [%]")

legend_handles = [Line2D([0], [0], marker=marker, color=(0, 0, 0, 0), label=model, 
                         markerfacecolor='darkgrey', markeredgecolor='darkgrey', markersize=8) 
                  for model, marker in dict_cmip6_marker.items()]
fig.legend(handles=legend_handles,loc="center left", bbox_to_anchor=(0.83, 0.5), frameon=False)
ax1.spines["top"].set_visible(False)
ax1.spines["right"].set_visible(False)
ax1.spines["bottom"].set_visible(False)
ax1.tick_params(axis="x", which="both", bottom=True, labelbottom=True)
ax2.spines["top"].set_visible(False)
ax2.spines["right"].set_visible(False)
fig.subplots_adjust(right=0.8, hspace=0.15)
#fig.savefig('../plots/cmip6_amoc&tas_scatter_2100.pdf', bbox_inches='tight', transparent=True)

In [ ]:
fig, ax2 = plt.subplots(figsize=(9, 10))
for i, sce in enumerate(["ssp126", "ssp245", "ssp370"]):
    for model in sce_tas_eur_cmip6[sce]:
        if (model in historical_amoc_cmip6 and model in sce_tas_eur_cmip6[sce] and model in historical_tas_cmip6):
            amoc_ref, tas_eur_ref, tas_glo_ref = [], [], []
            amoc_2100, tas_eur_2100, tas_glo_2100 = [], [], []
            for rea in sce_tas_eur_cmip6[sce][model]:
                if (rea in historical_amoc_cmip6[model] and rea in sce_tas_eur_cmip6[sce][model] and rea in historical_tas_cmip6[model]):
                    amoc_ref.append(historical_amoc_cmip6[model][rea].amoc.isel(time=slice(0,50)).mean("time"))
                    amoc_2100.append(sce_amoc_cmip6[sce][model][rea].amoc.isel(time=slice(-10,None)).mean("time"))
                    tas_eur_ref.append(weighted_area_lat(historical_tas_eur_cmip6[model][rea]).mean('lat').mean('lon').tas.isel(time=slice(0,50)).mean("time"))
                    tas_eur_2100.append(weighted_area_lat(sce_tas_eur_cmip6[sce][model][rea]).mean('lat').mean('lon').tas.isel(time=slice(-10,None)).mean("time"))
                    tas_glo_ref.append(weighted_area_lat(historical_tas_cmip6[model][rea]).mean('lat').mean('lon').tas.isel(time=slice(0,50)).mean("time"))
                    tas_glo_2100.append(weighted_area_lat(sce_tas_cmip6[sce][model][rea]).mean('lat').mean('lon').tas.isel(time=slice(-10,None)).mean("time"))
            
            ax2.scatter(100*(np.mean(amoc_2100)-np.mean(amoc_ref))/np.mean(amoc_ref), 
                        (np.mean(tas_eur_2100)-np.mean(tas_eur_ref))/(np.mean(tas_glo_2100)-np.mean(tas_glo_ref)),
                        color=ssp_col[sce], marker  = dict_cmip6_marker[model], s=70)
    
    amoc_ref = his_amoc_yr.amoc.mean("realiz").isel(time=slice(0,50)).mean("time")
    amoc_2100 = ssp_amoc_yr.sel(scenar=sce).mean("realiz").amoc.isel(time=slice(-10,None)).mean("time")
    tas_eur_ref = weighted_area_lat(his_tas_eur_yr.mean("realiz")).mean('lat').mean('lon').tas.isel(time=slice(0,50)).mean("time")
    tas_eur_2100 = weighted_area_lat(ssp_tas_eur_yr.sel(scenar=sce).mean("realiz")).mean('lat').mean('lon').tas.isel(time=slice(-10,None)).mean("time")
    tas_glo_ref = weighted_area_lat(his_tas_yr.mean("realiz")).mean('lat').mean('lon').tas.isel(time=slice(0,50)).mean("time")
    tas_glo_2100 = weighted_area_lat(ssp_tas_yr.sel(scenar=sce).mean("realiz")).mean('lat').mean('lon').tas.isel(time=slice(-10,None)).mean("time")
 
    ax2.scatter(100*(amoc_2100-amoc_ref)/amoc_ref, (tas_eur_2100-tas_eur_ref)/(tas_glo_2100-tas_glo_ref),
                color=ssp_col[sce], marker = dict_cmip6_marker["mpi-esm1-2-lr"], s=70)
    
ax2.set_ylabel(r"$\Delta \mathrm{T}_{\mathrm{EU}}$/$\Delta \mathrm{T}_{\mathrm{GLOB}}$ in 2091-2100 w.r.t. 1850-1899 [°C]")
ax2.set_xlabel(r"$\Delta$AMOC strength in 2091-2100 w.r.t. 1850-1899 [%]")

legend_handles = [Line2D([0], [0], marker=marker, color=(0, 0, 0, 0), label=model, 
                         markerfacecolor='darkgrey', markeredgecolor='darkgrey', markersize=8) 
                  for model, marker in dict_cmip6_marker.items()]
fig.legend(handles=legend_handles,loc="center left", bbox_to_anchor=(0.83, 0.5), frameon=False)

ax2.spines["top"].set_visible(False)
ax2.spines["right"].set_visible(False)
fig.subplots_adjust(right=0.8, hspace=0.15)
fig.savefig('../plots/cmip6_amocdiffperc&tasratioglob_scatter_2100.pdf', bbox_inches='tight', transparent=True)

In [ ]:
models = ['ec-earth3', 'ipsl-cm6a-lr', 'ukesm1-0-ll','hadgem3-gc31-mm']
scenarios = ["ssp126", "ssp245", "ssp370"]

fig = plt.figure(figsize=(22, 34))
gs = GridSpec(nrows=len(models) + 2, ncols=4, width_ratios=[0.23, 1, 1, 1], 
              height_ratios=[0.18] + [1] * len(models) + [0.12], hspace=0.22, wspace=0.1, figure=fig)

header_ax = fig.add_subplot(gs[0, 0])
header_ax.axis("off")

for i, sce in enumerate(scenarios):
    ax = fig.add_subplot(gs[0, i + 1])
    ax.axis("off")
    ax.text(0.5,0.5, sce, ha="center", va="center", fontsize=24, fontweight="bold")

for m, model in enumerate(models):    
        row = m + 1
        label_ax = fig.add_subplot(gs[row, 0])
        label_ax.axis("off")
        label_ax.text(0.98, 0.5, model, ha="right", va="center", fontsize=24, fontweight="bold")
    
        for i, sce in enumerate(scenarios):
            if (model in historical_amoc_cmip6 and model in sce_tas_eur_cmip6[sce] and model in sce_tas_eur_cmip6[sce] and model in historical_tas_cmip6):
    
                amoc_ref, tas_eur_ref, tas_ref = [], [], []
                amoc_2100, tas_eur_2100, tas_2100 = [], [], []
                ax = fig.add_subplot(gs[row, i + 1], projection=ccrs.Robinson(), frameon=False)
        
                for rea in sce_tas_cmip6[sce][model]:
                    
                    if (rea in historical_amoc_cmip6[model] and rea in sce_tas_eur_cmip6[sce][model] and rea in historical_tas_cmip6[model]):

                        tas_eur_ref.append(convert360_180(historical_tas_eur_cmip6[model][rea].tas.isel(time=slice(0, 50)).mean("time")))
                        tas_eur_2100.append(convert360_180(sce_tas_eur_cmip6[sce][model][rea].tas.isel(time=slice(-10, None)).mean("time")))
                        tas_ref.append(convert360_180(historical_tas_cmip6[model][rea].tas.isel(time=slice(0, 50)).mean("time")))
                        tas_2100.append(convert360_180(sce_tas_cmip6[sce][model][rea].tas.isel(time=slice(-10, None)).mean("time")))
                        amoc_ref.append(historical_amoc_cmip6[model][rea].amoc.isel(time=slice(0, 50)).mean("time"))
                        amoc_2100.append(sce_amoc_cmip6[sce][model][rea].amoc.isel(time=slice(-10, None)).mean("time"))
        
                diff_eur = xr.concat(tas_eur_2100, dim="rea").mean("rea") - xr.concat(tas_eur_ref, dim="rea").mean("rea")
                diff = xr.concat(tas_2100, dim="rea").mean("rea") - xr.concat(tas_ref, dim="rea").mean("rea")
                neg_diff_eur = diff_eur.where(diff < 0)
                neg_diff = diff.where(diff < 0)
                if neg_diff_eur.notnull().any():
                    ax.add_feature(cfeature.LAND, facecolor='#b0b0b0') 
                    neg_plot=neg_diff.plot(transform=ccrs.PlateCarree(), ax=ax, vmin=-2, vmax=0, 
                                  cmap="Blues_r", add_colorbar=False)
        

                    ax.set_title(f"{-np.round((100 * (np.mean(amoc_2100) - np.mean(amoc_ref)) / np.mean(amoc_ref)),0):.0f}% weakening", fontsize=18, pad=14)
                    ax.coastlines()
                    ax.add_feature(cfeature.OCEAN, color="white", zorder=1)
                    ax.set_extent([-18.5, 38.5, 34.8, 72], crs=ccrs.PlateCarree())
                    add_square(ax, -13.5, 0,  60, 75.5, colour='white') # get rid of Nordic sea small islands
                    add_square(ax, -30, -18,  68, 75, colour='white') # get rid of Greenland

cbar_ax = fig.add_subplot(gs[-1, 1:])

plt.subplots_adjust(left=0.03, right=0.99, top=0.97, bottom=0.05)
plt.show()
#fig.savefig('../plots/cmip6_maps_netcooling.pdf', bbox_inches='tight', transparent=True)
#fig.savefig('../plots/cmip6_maps_netcooling.png', bbox_inches='tight')

In [ ]:
import matplotlib.colors as mcolors

base = plt.get_cmap("seismic")
n = 256

# Remove the central 20% of the colormap (white-ish transition zone)
sharp_colors = np.vstack([
    base(np.linspace(0.00, 0.4, n // 2)),
    base(np.linspace(0.6, 1.00, n // 2)),
])
sharp_seismic = mcolors.LinearSegmentedColormap.from_list("sharp_seismic", sharp_colors)

In [ ]:
models = ['cesm2', 'mri-esm2-0', 'noresm2-lm', 'noresm2-mm']
scenarios = ["ssp126", "ssp245"]

fig = plt.figure(figsize=(22, 34))
gs = GridSpec(nrows=len(models) + 2, ncols=3, width_ratios=[0.23, 1, 1], 
              height_ratios=[0.18] + [1] * len(models) + [0.12], hspace=0.22, wspace=0.1, figure=fig)

header_ax = fig.add_subplot(gs[0, 0])
header_ax.axis("off")

for i, sce in enumerate(scenarios):
    ax = fig.add_subplot(gs[0, i + 1])
    ax.axis("off")
    ax.text(0.5,0.5, sce, ha="center", va="center", fontsize=24, fontweight="bold")

for m, model in enumerate(models):    
        row = m + 1
        label_ax = fig.add_subplot(gs[row, 0])
        label_ax.axis("off")
        label_ax.text(0.98, 0.5, model, ha="right", va="center", fontsize=24, fontweight="bold")
    
        for i, sce in enumerate(scenarios):
            if (model in historical_amoc_cmip6 and model in sce_tas_eur_cmip6[sce] and model in sce_tas_eur_cmip6[sce] and model in historical_tas_cmip6):
    
                amoc_ref, tas_eur_ref, tas_ref = [], [], []
                amoc_2100, tas_eur_2100, tas_2100 = [], [], []
                ax = fig.add_subplot(gs[row, i + 1], projection=ccrs.Robinson(), frameon=False)
        
                for rea in sce_tas_cmip6[sce][model]:
                    
                    if (rea in historical_amoc_cmip6[model] and rea in sce_tas_eur_cmip6[sce][model] and rea in historical_tas_cmip6[model]):

                        tas_eur_ref.append(convert360_180(historical_tas_eur_cmip6[model][rea].tas.isel(time=slice(0, 50)).mean("time")))
                        tas_eur_2100.append(convert360_180(sce_tas_eur_cmip6[sce][model][rea].tas.isel(time=slice(-10, None)).mean("time")))
                        tas_ref.append(convert360_180(historical_tas_cmip6[model][rea].tas.isel(time=slice(0, 50)).mean("time")))
                        tas_2100.append(convert360_180(sce_tas_cmip6[sce][model][rea].tas.isel(time=slice(-10, None)).mean("time")))
                        amoc_ref.append(historical_amoc_cmip6[model][rea].amoc.isel(time=slice(0, 50)).mean("time"))
                        amoc_2100.append(sce_amoc_cmip6[sce][model][rea].amoc.isel(time=slice(-10, None)).mean("time"))
        
                diff_eur = xr.concat(tas_eur_2100, dim="rea").mean("rea") - xr.concat(tas_eur_ref, dim="rea").mean("rea")
                diff = xr.concat(tas_2100, dim="rea").mean("rea") - xr.concat(tas_ref, dim="rea").mean("rea")
                neg_diff_eur = diff_eur.where(diff < 0)
                neg_diff = diff.where(diff < 0)
                if neg_diff_eur.notnull().any():
                    ax.add_feature(cfeature.LAND, facecolor='#b0b0b0') 
                    neg_plot=neg_diff.plot(transform=ccrs.PlateCarree(), ax=ax, vmin=-2, vmax=0, 
                                  cmap="Blues_r", add_colorbar=False, levels=np.linspace(-2,2,21))
        

                    ax.set_title(f"{-np.round((100 * (np.mean(amoc_2100) - np.mean(amoc_ref)) / np.mean(amoc_ref)),0):.0f}% weakening", fontsize=18, pad=14)
                    ax.coastlines()
                    ax.add_feature(cfeature.OCEAN, color="white", zorder=1)
                    ax.set_extent([-18.5, 38.5, 34.8, 72], crs=ccrs.PlateCarree())
                    add_square(ax, -13.5, 0,  60, 75.5, colour='white') # get rid of Nordic sea small islands
                    add_square(ax, -30, -18,  68, 75, colour='white') # get rid of Greenland

cbar_ax = fig.add_subplot(gs[-1, 1:])
cbar = fig.colorbar(neg_plot, cax=cbar_ax, orientation="horizontal")
cbar.set_label("Cooling in 2091-2100 w.r.t. 1850-1899 [°C]", fontsize=22)
cbar.ax.tick_params(labelsize=16, length=3, width=0.8)

plt.subplots_adjust(left=0.03, right=0.99, top=0.97, bottom=0.05)
plt.show()
fig.savefig('../plots/cmip6_maps_netcooling.pdf', bbox_inches='tight', transparent=True)
fig.savefig('../plots/cmip6_maps_netcooling.png', bbox_inches='tight')

In [ ]:
models = ['cesm2', 'mri-esm2-0', 'noresm2-lm', 'noresm2-mm']
scenarios = ["ssp126", "ssp245"]

fig = plt.figure(figsize=(22, 34))
gs = GridSpec(nrows=len(models) + 2, ncols=3, width_ratios=[0.23, 1, 1], 
              height_ratios=[0.18] + [1] * len(models) + [0.12], hspace=0.22, wspace=0.1, figure=fig)

header_ax = fig.add_subplot(gs[0, 0])
header_ax.axis("off")

for i, sce in enumerate(scenarios):
    ax = fig.add_subplot(gs[0, i + 1])
    ax.axis("off")
    ax.text(0.5,0.5, sce, ha="center", va="center", fontsize=24, fontweight="bold")

for m, model in enumerate(models):    
        row = m + 1
        label_ax = fig.add_subplot(gs[row, 0])
        label_ax.axis("off")
        label_ax.text(0.98, 0.5, model, ha="right", va="center", fontsize=24, fontweight="bold")
    
        for i, sce in enumerate(scenarios):
            if (model in historical_amoc_cmip6 and model in sce_tas_eur_cmip6[sce] and model in sce_tas_eur_cmip6[sce] and model in historical_tas_cmip6):
    
                amoc_ref, tas_eur_ref, tas_ref = [], [], []
                amoc_2100, tas_eur_2100, tas_2100 = [], [], []
                ax = fig.add_subplot(gs[row, i + 1], projection=ccrs.Robinson(), frameon=False)
        
                for rea in sce_tas_cmip6[sce][model]:
                    
                    if (rea in historical_amoc_cmip6[model] and rea in sce_tas_eur_cmip6[sce][model] and rea in historical_tas_cmip6[model]):

                        tas_eur_ref.append(convert360_180(historical_tas_eur_cmip6[model][rea].tas.isel(time=slice(0, 50)).mean("time")))
                        tas_eur_2100.append(convert360_180(sce_tas_eur_cmip6[sce][model][rea].tas.isel(time=slice(-10, None)).mean("time")))
                        tas_ref.append(convert360_180(historical_tas_cmip6[model][rea].tas.isel(time=slice(0, 50)).mean("time")))
                        tas_2100.append(convert360_180(sce_tas_cmip6[sce][model][rea].tas.isel(time=slice(-10, None)).mean("time")))
                        amoc_ref.append(historical_amoc_cmip6[model][rea].amoc.isel(time=slice(0, 50)).mean("time"))
                        amoc_2100.append(sce_amoc_cmip6[sce][model][rea].amoc.isel(time=slice(-10, None)).mean("time"))
        
                diff_eur = xr.concat(tas_eur_2100, dim="rea").mean("rea") - xr.concat(tas_eur_ref, dim="rea").mean("rea")
                diff = xr.concat(tas_2100, dim="rea").mean("rea") - xr.concat(tas_ref, dim="rea").mean("rea")
                neg_diff_eur = diff_eur.where(diff < 0)
                neg_diff = diff#.where(diff < 0)
                if neg_diff_eur.notnull().any():
                    ax.add_feature(cfeature.LAND, facecolor='#b0b0b0') 
                    neg_plot=neg_diff.plot(transform=ccrs.PlateCarree(), ax=ax, vmin=-2, vmax=2, 
                                  cmap=sharp_seismic, add_colorbar=False, levels=np.linspace(-2,2,21))
        

                    ax.set_title(f"{-np.round((100 * (np.mean(amoc_2100) - np.mean(amoc_ref)) / np.mean(amoc_ref)),0):.0f}% weakening", fontsize=18, pad=14)
                    ax.coastlines()
                    ax.add_feature(cfeature.OCEAN, color="white", zorder=1)
                    ax.set_extent([-18.5, 38.5, 34.8, 72], crs=ccrs.PlateCarree())
                    add_square(ax, -13.5, 0,  60, 75.5, colour='white') # get rid of Nordic sea small islands
                    add_square(ax, -30, -18,  68, 75, colour='white') # get rid of Greenland

cbar_ax = fig.add_subplot(gs[-1, 1:])
cbar = fig.colorbar(neg_plot, cax=cbar_ax, orientation="horizontal")
cbar.set_label(r"$\Delta$T in 2091-2100 w.r.t. 1850-1899 [°C]", fontsize=22)
cbar.ax.tick_params(labelsize=16, length=3, width=0.8)

plt.subplots_adjust(left=0.03, right=0.99, top=0.97, bottom=0.05)
plt.show()
fig.savefig('../plots/cmip6_maps_netcooling_all.pdf', bbox_inches='tight', transparent=True)
fig.savefig('../plots/cmip6_maps_netcooling_all.png', bbox_inches='tight')

# NAHosMIP

siconc

In [ ]:
# Annual mean T
models = ["CanESM5", "CESM2", "EC-Earth3", "HadGEM3-GC3-1LL", "HadGEM3-GC3-1MM", "IPSL-CM6A-LR", "MPI-ESM1-2-HR", "MPI-ESM1-2-LR"]
path = "/work/uo1075/m300817/hosing/nahosmip/" 
siconc_nahosmip = {}
for model in models:
    ds =  xr.open_mfdataset(path+f"/{model}/u03-hos/*siconc*.nc", use_cftime=True, parallel=True)
    if 'lon' in ds.coords:
        lon_coord = 'lon'
    elif 'longitude' in ds.coords:
        lon_coord = 'longitude'
    else:
        # Handle case where longitude coordinate has a different name
        lon_coord = None
        for coord in ds.coords:
            if 'lon' in coord.lower():
                lon_coord = coord
                break
    
    if lon_coord and ds[lon_coord].max() > 180:
        # Convert from 0-360 to -180-180
        ds = ds.assign_coords({lon_coord: (ds[lon_coord] + 180) % 360 - 180})
        ds = ds.sortby(lon_coord)
    
    siconc_nahosmip[model] = ds

In [ ]:
possible_var_names = ['siconc', 'aice', 'soicecov']   # Define possible variable names for sea ice concentration
possible_time_names = ['time', 'time_counter']   # Define possible time dimension names

def get_siconc_variable(ds, possible_names):
    """Find the sea ice concentration variable in the dataset"""
    for var_name in possible_names:
        if var_name in ds.variables:
            return var_name
    
    # If none found, look for variables with 'ice' and 'conc' in the name
    for var_name in ds.variables:
        if 'ice' in var_name.lower() and 'conc' in var_name.lower():
            return var_name
    
    # If still nothing, look for any variable with 'ice' in the name
    for var_name in ds.variables:
        if 'ice' in var_name.lower():
            return var_name
    
    # Last resort: print available variables and return None
    print(f"Could not find sea ice concentration variable. Available variables: {list(ds.variables.keys())}")
    return None

def get_time_dimension(ds, possible_names):
    """Find the time dimension in the dataset"""
    for time_name in possible_names:
        if time_name in ds.dims:
            return time_name
    
    # If none found, look for dimensions with 'time' in the name
    for dim_name in ds.dims:
        if 'time' in dim_name.lower():
            return dim_name
    
    # Last resort: print available dimensions and return None
    print(f"Could not find time dimension. Available dimensions: {list(ds.dims.keys())}")
    return None

In [ ]:
# Define North Atlantic Arctic region bounds
lat_min, lat_max = 40, 90
lon_min, lon_max = -80, 55

# Create subplot figure
fig = plt.figure(figsize=(20, 13))
n_cols = 3
n_rows = 3
for i, model in enumerate(models):
    ds = siconc_nahosmip[model]
    var_name = get_siconc_variable(ds, possible_var_names)
    time_dim = get_time_dimension(ds, possible_time_names)
    ds_region = ds.sel(lat=slice(lat_min, lat_max), lon=slice(lon_min, lon_max))
    ds_last5 = ds_region[var_name].isel({time_dim: slice(95, 100)}).mean(dim=time_dim)
    ds_first5 = ds_region[var_name].isel({time_dim: slice(0, 5)}).mean(dim=time_dim)
    year = 0
    siconc_plot = ds_region[var_name].isel({time_dim: year})
    
    ax = fig.add_subplot(n_rows, n_cols, i+1, projection=ccrs.PlateCarree())

    im = siconc_plot.plot(ax=ax, transform=ccrs.PlateCarree(), cmap='Blues_r', 
                          vmin=0, vmax=1, levels=np.linspace(0.,1,21), add_colorbar=False)
    
    #ax.coastlines()
    ax.add_feature(cfeature.LAND, color='lightgray')
    ax.add_feature(cfeature.OCEAN, color='white')
    
    ax.set_extent([lon_min, lon_max, lat_min, lat_max], ccrs.PlateCarree())
    ax.set_title(f'{model}', fontsize=10, y=0.95)
    ax.set_frame_on(False)
cbar_ax = fig.add_axes([0.92, 0.2, 0.02, 0.6])
cbar = fig.colorbar(im, cax=cbar_ax, ticks=[0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1])
cbar.set_label('Sea ice concentration', rotation=270, labelpad=20)

plt.tight_layout()
plt.subplots_adjust(hspace=-0.6, right=0.9)
plt.suptitle(f"Year {year+1}", y=0.85)
plt.show()

In [ ]:
# Define North Atlantic Arctic region bounds
lat_min, lat_max = 40, 90
lon_min, lon_max = -80, 55

# Create subplot figure
fig = plt.figure(figsize=(20, 13))
n_cols = 3
n_rows = 3
for i, model in enumerate(models):
    ds = siconc_nahosmip[model]
    var_name = get_siconc_variable(ds, possible_var_names)
    time_dim = get_time_dimension(ds, possible_time_names)
    ds_region = ds.sel(lat=slice(lat_min, lat_max), lon=slice(lon_min, lon_max))
    ds_last5 = ds_region[var_name].isel({time_dim: slice(95, 100)}).mean(dim=time_dim)
    ds_first5 = ds_region[var_name].isel({time_dim: slice(0, 5)}).mean(dim=time_dim)
    year = 9
    siconc_plot = ds_region[var_name].isel({time_dim: year})
    
    ax = fig.add_subplot(n_rows, n_cols, i+1, projection=ccrs.PlateCarree())

    im = siconc_plot.plot(ax=ax, transform=ccrs.PlateCarree(), cmap='Blues_r', 
                          vmin=0, vmax=1, levels=np.linspace(0.,1,21), add_colorbar=False)
    
    #ax.coastlines()
    ax.add_feature(cfeature.LAND, color='lightgray')
    ax.add_feature(cfeature.OCEAN, color='white')
    
    ax.set_extent([lon_min, lon_max, lat_min, lat_max], ccrs.PlateCarree())
    ax.set_title(f'{model}', fontsize=10, y=0.95)
    ax.set_frame_on(False)
cbar_ax = fig.add_axes([0.92, 0.2, 0.02, 0.6])
cbar = fig.colorbar(im, cax=cbar_ax, ticks=[0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1])
cbar.set_label('Sea ice concentration', rotation=270, labelpad=20)

plt.tight_layout()
plt.subplots_adjust(hspace=-0.6, right=0.9)
plt.suptitle(f"Year {year+1}", y=0.85)
plt.show()

In [ ]:
# Define North Atlantic Arctic region bounds
lat_min, lat_max = 40, 90
lon_min, lon_max = -80, 55

# Create subplot figure
fig = plt.figure(figsize=(20, 13))
n_cols = 3
n_rows = 3
for i, model in enumerate(models):
    ds = siconc_nahosmip[model]
    var_name = get_siconc_variable(ds, possible_var_names)
    time_dim = get_time_dimension(ds, possible_time_names)
    ds_region = ds.sel(lat=slice(lat_min, lat_max), lon=slice(lon_min, lon_max))
    ds_last5 = ds_region[var_name].isel({time_dim: slice(95, 100)}).mean(dim=time_dim)
    ds_first5 = ds_region[var_name].isel({time_dim: slice(0, 5)}).mean(dim=time_dim)
    year = 99
    siconc_plot = ds_region[var_name].isel({time_dim: year})
    
    ax = fig.add_subplot(n_rows, n_cols, i+1, projection=ccrs.PlateCarree())

    im = siconc_plot.plot(ax=ax, transform=ccrs.PlateCarree(), cmap='Blues_r', 
                          vmin=0, vmax=1, levels=np.linspace(0.,1,21), add_colorbar=False)
    
    #ax.coastlines()
    ax.add_feature(cfeature.LAND, color='lightgray')
    ax.add_feature(cfeature.OCEAN, color='white')
    
    ax.set_extent([lon_min, lon_max, lat_min, lat_max], ccrs.PlateCarree())
    ax.set_title(f'{model}', fontsize=10, y=0.95)
    ax.set_frame_on(False)
cbar_ax = fig.add_axes([0.92, 0.2, 0.02, 0.6])
cbar = fig.colorbar(im, cax=cbar_ax, ticks=[0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1])
cbar.set_label('Sea ice concentration', rotation=270, labelpad=20)

plt.tight_layout()
plt.subplots_adjust(hspace=-0.6, right=0.9)
plt.suptitle(f"Year {year+1}", y=0.85)
plt.show()

In [ ]:
# Define North Atlantic Arctic region bounds
lat_min, lat_max = 40, 90
lon_min, lon_max = -80, 55

# Create subplot figure
fig = plt.figure(figsize=(20, 13))
n_cols = 3
n_rows = 3
for i, model in enumerate(models):
    ds = siconc_nahosmip[model]
    var_name = get_siconc_variable(ds, possible_var_names)
    time_dim = get_time_dimension(ds, possible_time_names)
    ds_region = ds.sel(lat=slice(lat_min, lat_max), lon=slice(lon_min, lon_max))
    ds_last5 = ds_region[var_name].isel({time_dim: slice(95, 100)}).mean(dim=time_dim)
    ds_first5 = ds_region[var_name].isel({time_dim: slice(0, 5)}).mean(dim=time_dim)
    year = 99
    siconc_plot = ds_region[var_name].isel({time_dim: year})
    
    ax = fig.add_subplot(n_rows, n_cols, i+1, projection=ccrs.PlateCarree())

    im = siconc_plot.plot(ax=ax, transform=ccrs.PlateCarree(), cmap='Blues_r', 
                          vmin=0, vmax=1, levels=np.linspace(0.,1,21), add_colorbar=False)
    
    #ax.coastlines()
    ax.add_feature(cfeature.LAND, color='lightgray')
    ax.add_feature(cfeature.OCEAN, color='white')
    
    ax.set_extent([lon_min, lon_max, lat_min, lat_max], ccrs.PlateCarree())
    ax.set_title(f'{model}', fontsize=10, y=0.95)
    ax.set_frame_on(False)
cbar_ax = fig.add_axes([0.92, 0.2, 0.02, 0.6])
cbar = fig.colorbar(im, cax=cbar_ax, ticks=[0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1])
cbar.set_label('Sea ice concentration', rotation=270, labelpad=20)

plt.tight_layout()
plt.subplots_adjust(hspace=-0.6, right=0.9)
plt.suptitle(f"Year {year+1}", y=0.85)
plt.show()

# MPI-GE

siarean

In [ ]:
path = '/work/uo1075/m300817/teu_amoc/data/MPI-GE/'
his_siarean_march = xr.open_mfdataset(path+"his_siarean_march.nc", use_cftime=True, parallel=True)
his_siarean_sept = xr.open_mfdataset(path+"his_siarean_sept.nc", use_cftime=True, parallel=True)
ssp_siarean_march = xr.open_mfdataset(path+"ssp_siarean_march.nc", use_cftime=True, parallel=True)
ssp_siarean_sept = xr.open_mfdataset(path+"ssp_siarean_sept.nc", use_cftime=True, parallel=True)

his_siarean_yr = xr.open_mfdataset(path+"his_siarean_yr.nc", use_cftime=True, parallel=True)
ssp_siarean_yr = xr.open_mfdataset(path+"ssp_siarean_yr.nc", use_cftime=True, parallel=True)

In [ ]:
fig = plt.figure(figsize=(10, 10))
gs = GridSpec(1, 1)
ax1 = fig.add_subplot(gs[0, 0])
for exp in ['ssp126','ssp245', 'ssp370']: 
    time = ssp_siarean_yr.sel(scenar=exp).__xarray_dataarray_variable__.rolling(time=10, center=True).mean("time").time.dt.year
    siarean = ssp_siarean_yr.sel(scenar=exp).__xarray_dataarray_variable__.rolling(time=10, center=True).mean("time")
    ax1.plot(time, siarean.mean("realiz"), label=exp, alpha=0.8, linewidth=3, color=ssp_col[exp])
    ax1.fill_between(time, -siarean.std("realiz")+siarean.mean("realiz"), siarean.std("realiz")+siarean.mean("realiz"), 
                     alpha=0.5, color=ssp_col[exp])
ax1.set_xlim(2015, 2100)
ax1.set_xlabel("Time [year]")
ax1.set_ylabel(r"Arctic sea ice area [km$^2$]")
ax1.spines['right'].set_visible(False)
ax1.spines['top'].set_visible(False)
ax1.spines['left'].set_position(('outward', 10))  # Move the left spine outward by 10 points
ax1.spines['bottom'].set_position(('outward', 10)) 

plt.legend(loc=3)
plt.show()
fig.savefig('../plots/MPI-GE_siarean_yr.png', bbox_inches='tight')

In [ ]:
ssp_siarean_sept.sel(scenar=exp).siarean.isel(time=slice(-10, None)).mean("time").mean("realiz").values


In [ ]:
for exp in ['ssp126', 'ssp245', 'ssp370']: 
    siarean = ssp_siarean_sept.sel(scenar=exp).siarean.isel(time=slice(-10, None)).mean("time").mean("realiz").values
    print(siarean)
    siarean = ssp_siarean_sept.sel(scenar=exp).siarean.isel(time=slice(-10, None)).mean("time").std("realiz").values
    print(siarean)

In [ ]:
his_siarean_sept_ext = xr.concat([his_siarean_sept, ssp_siarean_sept.sel(scenar='ssp245').isel(time=slice(0, 9))], dim="time")
his_siarean_march_ext = xr.concat([his_siarean_march, ssp_siarean_march.sel(scenar='ssp245').isel(time=slice(0, 9))], dim="time")

fig = plt.figure(figsize=(10, 10))
gs = GridSpec(1, 1)
ax1 = fig.add_subplot(gs[0, 0])
time = his_siarean_sept_ext.siarean.rolling(time=10, center=True).mean("time").time.dt.year
siarean = his_siarean_sept_ext.siarean.rolling(time=10, center=True).mean("time")
ax1.plot(time, siarean.mean("realiz"), label='historical', alpha=0.8, linewidth=3, color='black')
ax1.fill_between(time, -siarean.std("realiz")+siarean.mean("realiz"), siarean.std("realiz")+siarean.mean("realiz"), 
                 alpha=0.5, color='black')
siarean = his_siarean_march_ext.siarean.rolling(time=10, center=True).mean("time")
ax1.plot(time, siarean.mean("realiz"), alpha=0.8, linewidth=3, color='black')
ax1.fill_between(time, -siarean.std("realiz")+siarean.mean("realiz"), siarean.std("realiz")+siarean.mean("realiz"), 
                 alpha=0.5, color='black')
for exp in ['ssp126','ssp245', 'ssp370']: 
    time = ssp_siarean_sept.sel(scenar=exp).siarean.rolling(time=10, center=True).mean("time").time.dt.year
    siarean = ssp_siarean_sept.sel(scenar=exp).siarean.rolling(time=10, center=True).mean("time")
    ax1.plot(time, siarean.mean("realiz"), alpha=0.8, linewidth=3, color=ssp_col[exp])
    ax1.fill_between(time, -siarean.std("realiz")+siarean.mean("realiz"), siarean.std("realiz")+siarean.mean("realiz"), 
                     alpha=0.5, color=ssp_col[exp])
    siarean = ssp_siarean_march.sel(scenar=exp).siarean.rolling(time=10, center=True).mean("time")
    ax1.plot(time, siarean.mean("realiz"), label=exp, alpha=0.8, linewidth=3, color=ssp_col[exp])
    ax1.fill_between(time, -siarean.std("realiz")+siarean.mean("realiz"), siarean.std("realiz")+siarean.mean("realiz"), 
                     alpha=0.5, color=ssp_col[exp])
ax1.set_xlim(1850, 2100)
ax1.set_xlabel("Time [year]")
ax1.set_ylabel(r"Arctic sea ice area [km$^2$]")
ax1.spines['right'].set_visible(False)
ax1.spines['top'].set_visible(False)
ax1.spines['left'].set_position(('outward', 10))  # Move the left spine outward by 10 points
ax1.spines['bottom'].set_position(('outward', 10)) 
plt.text(0.44, 0.77, 'March', transform=ax1.transAxes, fontsize=18, fontweight='bold')
plt.text(0.4, 0.18, 'September', transform=ax1.transAxes, fontsize=18, fontweight='bold')
plt.legend(loc=10)
plt.show()
fig.savefig('../plots/MPI-GE_siarean.pdf', bbox_inches='tight', transparent=True)
fig.savefig('../plots/MPI-GE_siarean.png', bbox_inches='tight')

siconc

In [ ]:
path = '/work/uo1075/m300817/teu_amoc/data/MPI-GE/'
#his_siconc_mon = xr.open_mfdataset(path+"his_siconc_mon.nc", use_cftime=True, parallel=True)
ssp_siconc_yr = xr.open_mfdataset(path+"ssp_siconc_yr.nc", use_cftime=True, parallel=True)
ssp_siconc_mon = xr.open_mfdataset(path+"ssp_siconc_mon.nc", use_cftime=True, parallel=True)
ssp_siconc_march = xr.open_mfdataset(path+"ssp_siconc_march.nc", use_cftime=True, parallel=True)

In [ ]:
ds = ssp_siconc_yr
lat_min, lat_max = 40, 90
lon_min, lon_max = -80, 55
# Create subplot figure
fig = plt.figure(figsize=(12, 12))
n_cols = 1
n_rows = 3
ds_region = ds#.where(mask, drop=True)#.sel(latitude=slice(lat_min, lat_max), longitude=slice(lon_min, lon_max))
siconc_plot = ds_region.__xarray_dataarray_variable__.isel(time=slice(-10, None)).mean("time")
for i, sce in enumerate(['ssp126','ssp245', 'ssp370']):  
    ax = fig.add_subplot(n_rows, n_cols, i+1, projection=ccrs.PlateCarree())
    im = siconc_plot.sel(scenar=sce).mean("realiz").plot(ax=ax, transform=ccrs.PlateCarree(), cmap='Blues_r', 
                              x="longitude", y="latitude", vmin=0, vmax=1, levels=np.linspace(0.,1,21), add_colorbar=False, extend="neither")
    ax.add_feature(cfeature.LAND, color='lightgray')
    ax.add_feature(cfeature.OCEAN, color='white')
    ax.set_extent([lon_min, lon_max, lat_min, lat_max], ccrs.PlateCarree())
    ax.set_title(f'{sce}', fontsize=20, y=1, color=ssp_col[sce])
    ax.set_frame_on(False)
cbar_ax = fig.add_axes([0.92, 0.2, 0.03, 0.6]) # left, bottom, width, height
cbar = fig.colorbar(im, cax=cbar_ax, ticks=[0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1])
cbar.set_label('Sea ice concentration', rotation=270, labelpad=20)
plt.tight_layout()
plt.subplots_adjust(hspace=0.3)
plt.show()
#fig.savefig('../plots/MPI-GE_siconc.pdf', bbox_inches='tight', transparent=True)
fig.savefig('../plots/MPI-GE_siconc_yr.png', bbox_inches='tight')

In [ ]:
lat_min, lat_max = 40, 90
lon_min, lon_max = -80, 55
fig = plt.figure(figsize=(16, 12))
n_cols = 2
n_rows = 3
fig.text(0.231, 1.05, 'March', ha='center', va='top', fontsize=24, fontweight="bold")
fig.text(0.679, 1.05, 'September', ha='center', va='top', fontsize=24, fontweight="bold")

for i, sce in enumerate(['ssp126','ssp245', 'ssp370']):  
    ax = fig.add_subplot(n_rows, n_cols, 2*i+1, projection=ccrs.PlateCarree())
    ds = ssp_siconc_march
    siconc_plot = ds.siconc.isel(time=slice(-10, None)).mean("time")
    im = siconc_plot.sel(scenar=sce).mean("realiz").plot(ax=ax, transform=ccrs.PlateCarree(), cmap='Blues_r', 
                              x="longitude", y="latitude", vmin=0, vmax=1, levels=np.linspace(0.,1,21), add_colorbar=False, extend="neither")
    ax.add_feature(cfeature.LAND, color='lightgray')
    ax.add_feature(cfeature.OCEAN, color='white')
    ax.set_extent([lon_min, lon_max, lat_min, lat_max], ccrs.PlateCarree())
    ax.set_title(f'{sce}', fontsize=20, y=1, color=ssp_col[sce])
    ax.set_frame_on(False)

    ds = ssp_siconc_mon.where(ssp_siconc_mon.time.dt.month == 9, drop=True)
    siconc_plot = ds.siconc.isel(time=slice(-10, None)).mean("time")
    ax = fig.add_subplot(n_rows, n_cols, 2*i+2, projection=ccrs.PlateCarree())
    im = siconc_plot.sel(scenar=sce).mean("realiz").plot(ax=ax, transform=ccrs.PlateCarree(), cmap='Blues_r', 
                              x="longitude", y="latitude", vmin=0, vmax=1, levels=np.linspace(0.,1,21), add_colorbar=False, extend="neither")
    ax.add_feature(cfeature.LAND, color='lightgray')
    ax.add_feature(cfeature.OCEAN, color='white')
    ax.set_extent([lon_min, lon_max, lat_min, lat_max], ccrs.PlateCarree())
    ax.set_title(f'{sce}', fontsize=19, y=1, color=ssp_col[sce])
    ax.set_frame_on(False)
cbar_ax = fig.add_axes([0.92, 0.2, 0.03, 0.6]) # left, bottom, width, height
cbar = fig.colorbar(im, cax=cbar_ax, ticks=[0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1])
cbar.set_label('Sea ice concentration', rotation=270, labelpad=20)
plt.tight_layout()
plt.subplots_adjust(hspace=0.1, right=0.9)
plt.show()
fig.savefig('../plots/MPI-GE_siconc.pdf', bbox_inches='tight', transparent=True)
fig.savefig('../plots/MPI-GE_siconc.png', bbox_inches='tight')

combined

In [ ]:
fig = plt.figure(figsize=(22, 10))
gs = GridSpec(3, 2, figure=fig, width_ratios=[1.2, 1.0], wspace=0, hspace=0.25)

ds = ssp_siconc_yr
lat_min, lat_max = 40, 90
lon_min, lon_max = -80, 55
siconc_plot = ds.__xarray_dataarray_variable__.isel(time=slice(-10, None)).mean("time")
panel_labels = ["(a)", "(b)", "(c)", "(d)"]
for i, sce in enumerate(["ssp126", "ssp245", "ssp370"]):
    ax = fig.add_subplot(gs[i, 1], projection=ccrs.PlateCarree())
    im = siconc_plot.sel(scenar=sce).mean("realiz").plot(ax=ax, transform=ccrs.PlateCarree(), cmap="Blues_r",
                                                         x="longitude", y="latitude", vmin=0,vmax=1, 
                                                         levels=np.linspace(0., 1, 21), add_colorbar=False, extend="neither")
    ax.add_feature(cfeature.LAND, color="lightgray")
    ax.add_feature(cfeature.OCEAN, color="white")
    ax.set_extent([lon_min, lon_max, lat_min, lat_max], ccrs.PlateCarree())
    ax.set_title(sce, fontsize=20, color=ssp_col[sce], y=0.93)
    ax.set_frame_on(False)
    ax.text(0.01, 0.98, panel_labels[i+1], transform=ax.transAxes, fontsize=14, fontweight="bold", ha="left", va="top",
            bbox=dict(boxstyle="round,pad=0.2", facecolor="white", alpha=0.7, edgecolor="none"))

# Right column: sea ice area spanning all rows
ax1 = fig.add_subplot(gs[:, 0])
for exp in ["ssp126", "ssp245", "ssp370"]:
    siarean = (ssp_siarean_yr.sel(scenar=exp).__xarray_dataarray_variable__.rolling(time=10, center=True).mean("time"))
    time = siarean.time.dt.year
    mean = siarean.mean("realiz")
    std = siarean.std("realiz")
    ax1.plot(time, mean, label=exp, alpha=0.8, linewidth=3, color=ssp_col[exp])
    ax1.fill_between(time, mean - std, mean + std, alpha=0.5, color=ssp_col[exp])

ax1.set_xlim(2015, 2100)
ax1.set_xlabel("Time [year]")
ax1.set_ylabel(r"Arctic sea ice area [10$^6$ km$^2$]")
ax1.spines["right"].set_visible(False)
ax1.spines["top"].set_visible(False)
ax1.spines["left"].set_position(("outward", 10))
ax1.spines["bottom"].set_position(("outward", 10))
ax1.legend(loc="lower left")
ax1.text(-0.11, 1.05, panel_labels[0], transform=ax1.transAxes, fontsize=14, fontweight="bold", ha="left", va="top", 
         bbox=dict(boxstyle="round,pad=0.2", facecolor="white", alpha=0.7, edgecolor="none"))

cbar_ax = fig.add_axes([0.89, 0.2, 0.03, 0.5])
cbar = fig.colorbar(im, cax=cbar_ax, ticks=np.linspace(0, 1, 11))
cbar.set_label("Sea ice concentration", rotation=270, labelpad=20)
plt.subplots_adjust(right=0.9, hspace=0.15)
plt.show()
fig.savefig('../plots/MPI-GE_si_combined.png', bbox_inches='tight')

# ERA5

In [ ]:
first_plot = True
for rea in his_tas_yr.realiz:
    ref19601989_gehiseur = weighted_area_lat(his_tas_eur_yr.sel(realiz=rea)).mean('lat').mean('lon').sel(
        time=slice(cftime.DatetimeProlepticGregorian(1960, 1, 1),cftime.DatetimeProlepticGregorian(1989, 12, 31))).mean('time').tas.values
    label = 'MPI-GE historical' if first_plot else None
    (weighted_area_lat(his_tas_eur_yr.sel(realiz=rea)).mean('lat').mean('lon').tas-ref19601989_gehiseur).plot(color='darkgrey', label=label, linewidth=2)
    first_plot = False
ref19601989_eraeur =weighted_area_lat(era5_tas_eur_yr).mean('lat').mean('lon').sel(
    time=slice(cftime.DatetimeProlepticGregorian(1960, 1, 1),cftime.DatetimeProlepticGregorian(1989, 12, 31))).mean('time').tas.values
(weighted_area_lat(era5_tas_eur_yr).mean('lat').mean('lon').tas-ref19601989_eraeur).plot(color='royalblue', label ='ERA5', linewidth=3)
plt.xlabel("time [year]")
plt.xlim([cftime.DatetimeProlepticGregorian(1940, 1, 1),cftime.DatetimeProlepticGregorian(2014, 1, 31)])
plt.legend(loc=9, frameon=False)
plt.ylabel(r"$\Delta \mathrm{T}_{\mathrm{EU}}$ [°C]")
ax = plt.gca()
ax.spines['right'].set_visible(False)
ax.spines['top'].set_visible(False)
ax.spines['left'].set_position(('outward', 10))  # Move the left spine outward by 10 points
ax.spines['bottom'].set_position(('outward', 10)) 

In [ ]:
hisref1960 = weighted_area_lat(his_tas_eur_yr).mean('lat').mean('lon').mean("realiz").sel(time=slice(cftime.DatetimeProlepticGregorian(1960, 1, 1),cftime.DatetimeProlepticGregorian(1989, 12, 31))).mean('time').tas.values
his_tas_eur_yr_anom = weighted_area_lat(his_tas_eur_yr).mean('lat').mean('lon').tas.sel(
        time=slice(cftime.DatetimeProlepticGregorian(1940, 1, 1),cftime.DatetimeProlepticGregorian(2014, 12, 31)))-hisref1960
eraref1960 = weighted_area_lat(era5_tas_eur_yr).mean('lat').mean('lon').sel(time=slice(cftime.DatetimeProlepticGregorian(1960, 1, 1),cftime.DatetimeProlepticGregorian(1989, 12, 31))).mean('time').tas.values
era5_tas_eur_yr_anom = weighted_area_lat(era5_tas_eur_yr).mean('lat').mean('lon').sel(
        time=slice(cftime.DatetimeProlepticGregorian(1940, 1, 1),cftime.DatetimeProlepticGregorian(2014, 12, 31))).tas-eraref1960

In [ ]:
fig = plt.figure(figsize=(12, 10))

hisref1960 = weighted_area_lat(his_tas_eur_yr).mean('lat').mean('lon').mean("realiz").sel(time=slice(cftime.DatetimeProlepticGregorian(1960, 1, 1),cftime.DatetimeProlepticGregorian(1989, 12, 31))).mean('time').tas.values
his_tas_eur_yr_anom = weighted_area_lat(his_tas_eur_yr).mean('lat').mean('lon').tas.sel(
        time=slice(cftime.DatetimeProlepticGregorian(1940, 1, 1),cftime.DatetimeProlepticGregorian(2014, 12, 31)))-hisref1960
eraref1960 = weighted_area_lat(era5_tas_eur_yr).mean('lat').mean('lon').sel(time=slice(cftime.DatetimeProlepticGregorian(1960, 1, 1),cftime.DatetimeProlepticGregorian(1989, 12, 31))).mean('time').tas.values
era5_tas_eur_yr_anom = weighted_area_lat(era5_tas_eur_yr).mean('lat').mean('lon').sel(
        time=slice(cftime.DatetimeProlepticGregorian(1940, 1, 1),cftime.DatetimeProlepticGregorian(2014, 12, 31))).tas-eraref1960

ranks_era = np.zeros_like(era5_tas_eur_yr_anom.values)
for i, tas_era in enumerate(era5_tas_eur_yr_anom.values):
    combined = [tas_era]
    for rea in his_tas_eur_yr_anom.realiz:
        combined.append(float(his_tas_eur_yr_anom.sel(realiz=rea).isel(time=i).values))
    ranked = stats.rankdata(combined, method='average')
    ranks_era[i] = ranked[0]

freq_table = np.zeros((50, 50)) # rows are experiments/realiz, columns are the ranks, values are the frequencies
rank_tas_realiz = his_tas_eur_yr_anom.rank(dim='realiz')
for i, rea in enumerate(his_tas_eur_yr_anom.realiz):
    freq, bin_edges = np.histogram(rank_tas_realiz.sel(realiz=rea), bins=np.arange(1, 52)-0.5)
    freq_table[i, :] = freq
    
freq_model10 = []
freq_model90 = []
for j, rea in enumerate(his_tas_eur_yr_anom.realiz):
    freq_model10.append(np.quantile(freq_table[:, j], 0.1))
    freq_model90.append(np.quantile(freq_table[:, j], 0.9))

hist, bin_edges = np.histogram(ranks_era, bins=np.arange(1, 52)-0.5)
plt.hist(ranks_era, bins=np.arange(52)-0.5, alpha=0.75, label='ERA5 histograms', color='skyblue')

mean_freq = np.convolve(hist, np.ones(7)/7, mode='valid')
plt.plot(np.arange(4, 48), mean_freq, alpha=0.85, color='royalblue', label='ERA5 7-bin window histogram slope', linewidth=4)

plt.plot(np.arange(4, 48), np.convolve(freq_model10, np.ones(7)/7, mode='valid'), 
         linestyle='dashed',  color='black', linewidth=4)
plt.plot(np.arange(4, 48), np.convolve(freq_model90, np.ones(7)/7, mode='valid'),
         linestyle='dashed',  color='black', linewidth=4, label='Perfect model 7-bin window histogram slope range (central p80)')
plt.xlabel('Rank')
plt.ylabel('Frequency')
plt.xlim(0,51)
plt.legend(fontsize=18, loc=9, bbox_to_anchor=(0.48, 1.2), frameon=False)
ax = plt.gca()
ax.spines['right'].set_visible(False)
ax.spines['top'].set_visible(False)
ax.spines['left'].set_position(('outward', 10))  # Move the left spine outward by 10 points
ax.spines['bottom'].set_position(('outward', 10)) 
plt.show()

# Supplementary

In [ ]:
ge_paths= {'historical'    : "/pool/data/CMIP6/data/CMIP/MPI-M/MPI-ESM1-2-LR/historical",
           'piControl'     : "/pool/data/CMIP6/data/CMIP/MPI-M/MPI-ESM1-2-LR/piControl/r1i1p1f1",
           'ssp126'        : '/pool/data/CMIP6/data/ScenarioMIP/MPI-M/MPI-ESM1-2-LR/ssp126',
           'ssp245'        : '/pool/data/CMIP6/data/ScenarioMIP/MPI-M/MPI-ESM1-2-LR/ssp245',
           'ssp370'        : '/pool/data/CMIP6/data/ScenarioMIP/MPI-M/MPI-ESM1-2-LR/ssp370',
           'ssp585'        : '/pool/data/CMIP6/data/ScenarioMIP/MPI-M/MPI-ESM1-2-LR/ssp585'
          }  

In [ ]:
ssphos_lst = []
for exp in ["ssp126", "ssp245", "ssp370"]:
    for forc in ["grc01Sv", "grc03Sv", "grc05Sv", "grclin02Sv", "grclin06Sv", "grclin10Sv", "grcneg01Sv", "grclinneg02Sv"]:
        ssphos_lst.append(f"{exp}_{forc}") 
realiz_lst = ["r1i1p1f1", "r2i1p1f1", "r3i1p1f1"]

In [ ]:
mpiesm_col = {
    'ssp126': (23/255, 60/255, 102/255), 
    'ssp126_grc01Sv': 'steelblue', 
    'ssp126_grc03Sv': 'cornflowerblue',
    'ssp126_grc05Sv': 'deepskyblue',
    'ssp126_grclin02Sv': 'cadetblue',
    'ssp126_grclin06Sv': 'mediumaquamarine', 
    'ssp126_grclin10Sv': 'aquamarine',
    'ssp126_grcneg01Sv': 'midnightblue',
    'ssp126_grclinneg02Sv': 'teal',
    
    'ssp245': (247/255, 148/255, 32/255), 
    'ssp245_grc01Sv': 'gold',
    'ssp245_grc03Sv': 'yellow',
    'ssp245_grc05Sv': 'navajowhite',
    'ssp245_grclin02Sv': 'darkkhaki',
    'ssp245_grclin06Sv': 'khaki',
    'ssp245_grclin10Sv': 'palegoldenrod', 
    'ssp245_grcneg01Sv': 'darkorange',
    'ssp245_grclinneg02Sv': 'olive',

    'ssp370': (231/255, 29/255, 37/255),
    'ssp370_grc01Sv': 'orangered',
    'ssp370_grc03Sv': 'salmon',
    'ssp370_grc05Sv': 'lightsalmon',
    'ssp370_grclin02Sv': 'hotpink', 
    'ssp370_grclin06Sv': 'plum',
    'ssp370_grclin10Sv': 'pink',
    'ssp370_grcneg01Sv': 'darkred',
    'ssp370_grclinneg02Sv': 'mediumorchid',
}

In [ ]:
def text_legend_hos(fonts=15):
    fig.text(0.2, 0.6, 'MPI-ESM GE historical', fontsize=fonts, color='black', weight='bold', va='top', ha='left', clip_on=False)
    
    fig.text(0.2, 0.58, 'MPI-ESM GE SSP3-7.0', fontsize=fonts, color=(231/255, 29/255, 37/255), weight='bold', va='top', ha='left', clip_on=False)
    fig.text(0.22, 0.56, '\u00A0-0.1Sv', fontsize=fonts, color='darkred', weight='bold', va='top', ha='left', clip_on=False)
    fig.text(0.32, 0.56, 'lin.\u00A0-0.2Sv', fontsize=fonts, color='mediumorchid', weight='bold', va='top', ha='left', clip_on=False)
    fig.text(0.22, 0.545, '+0.1Sv', fontsize=fonts, color='orangered', weight='bold', va='top', ha='left', clip_on=False)
    fig.text(0.32, 0.545, 'lin.+0.2Sv', fontsize=fonts, color='hotpink', weight='bold', va='top', ha='left', clip_on=False)
    fig.text(0.22, 0.530, '+0.3Sv', fontsize=fonts, color='salmon', weight='bold', va='top', ha='left', clip_on=False)
    fig.text(0.32, 0.530, 'lin.+0.6Sv', fontsize=fonts, color='plum', weight='bold', va='top', ha='left', clip_on=False)
    fig.text(0.22, 0.515, '+0.5Sv', fontsize=fonts, color='lightsalmon', weight='bold', va='top', ha='left', clip_on=False)
    fig.text(0.32, 0.515, 'lin.+1.0Sv', fontsize=fonts, color='pink', weight='bold', va='top', ha='left', clip_on=False)

    fig.text(0.2, 0.490, 'MPI-ESM GE SSP2-4.5', fontsize=fonts, color=(247/255, 148/255, 32/255), weight='bold', va='top', ha='left', clip_on=False)
    fig.text(0.22, 0.470, '\u00A0-0.1Sv', fontsize=fonts, color='darkorange', weight='bold', va='top', ha='left', clip_on=False)
    fig.text(0.32, 0.470, 'lin.\u00A0-0.2Sv', fontsize=fonts, color='olive', weight='bold', va='top', ha='left', clip_on=False)
    fig.text(0.22, 0.455, '+0.1Sv', fontsize=fonts, color='gold', weight='bold', va='top', ha='left', clip_on=False)
    fig.text(0.32, 0.455, 'lin.+0.2Sv', fontsize=fonts, color='darkkhaki', weight='bold', va='top', ha='left', clip_on=False)
    fig.text(0.22, 0.440, '+0.3Sv', fontsize=fonts, color='yellow', weight='bold', va='top', ha='left', clip_on=False)
    fig.text(0.32, 0.440, 'lin.+0.6Sv', fontsize=fonts, color='khaki', weight='bold', va='top', ha='left', clip_on=False)
    fig.text(0.22, 0.425, '+0.5Sv', fontsize=fonts, color='navajowhite', weight='bold', va='top', ha='left', clip_on=False)
    fig.text(0.32, 0.425, 'lin.+1.0Sv', fontsize=fonts, color='palegoldenrod', weight='bold', va='top', ha='left', clip_on=False)
    
    fig.text(0.2, 0.400, 'MPI-ESM GE SSP1-2.6', fontsize=fonts, color=(23/255, 60/255, 102/255), weight='bold', va='top', ha='left', clip_on=False)
    fig.text(0.22, 0.380, '\u00A0-0.1Sv', fontsize=fonts, color='midnightblue', weight='bold', va='top', ha='left', clip_on=False)
    fig.text(0.32, 0.380, 'lin.\u00A0-0.2Sv', fontsize=fonts, color='teal', weight='bold', va='top', ha='left', clip_on=False)
    fig.text(0.22, 0.365, '+0.1Sv', fontsize=fonts, color='steelblue', weight='bold', va='top', ha='left', clip_on=False)
    fig.text(0.32, 0.365, 'lin.+0.2Sv', fontsize=fonts, color='cadetblue', weight='bold', va='top', ha='left', clip_on=False)
    fig.text(0.22, 0.350, '+0.3Sv', fontsize=fonts, color='cornflowerblue', weight='bold', va='top', ha='left', clip_on=False)
    fig.text(0.32, 0.350, 'lin.+0.6Sv', fontsize=fonts, color='mediumaquamarine', weight='bold', va='top', ha='left', clip_on=False)
    fig.text(0.22, 0.335, '+0.5Sv', fontsize=fonts, color='deepskyblue', weight='bold', va='top', ha='left', clip_on=False)
    fig.text(0.32, 0.335, 'lin.+1.0Sv', fontsize=fonts, color='aquamarine', weight='bold', va='top', ha='left', clip_on=False)
    

In [ ]:
array_ssp = []
for exp in ['ssp126','ssp245', 'ssp370', 'ssp585']:      
    array_rea = []
    for i in range(1,51):
        rea = "r"+str(i)+"i1p1f1"
        if os.path.isdir(f'{ge_paths[exp]}/{rea}'):
            infiles = glob.glob(f'{ge_paths[exp]}/{rea}/Omon/msftmz/gn/**/*.nc', recursive=True)
            array_rea.append(xr.open_mfdataset(infiles, use_cftime=True, data_vars="minimal", coords="minimal", chunks={'time': 1980}, parallel=True, compat="override").assign_coords(realiz=rea)) # compat="override"
    array_ssp.append(xr.concat(array_rea, dim='realiz').assign_coords(scenar=exp))
ssp_msftmz_mon = xr.concat(array_ssp, dim='scenar')

In [ ]:
# AMOC strength 26.5N in Sv
ssp_amoc_yr = weighted_mon_to_year_mean(ssp_msftmz_mon.sel(basin=1, drop=True).sel(lat=26.5, drop=True), "msftmz")
max_lev_idx= ssp_amoc_yr.idxmax(dim='lev') #select depths with maximum msftmz
ssp_amoc_yr = (ssp_amoc_yr.sel(lev=max_lev_idx, drop=True))/(1025.0 * 10**6)

In [ ]:
# AMOC strength 26.5N in Sv
amoc_yr = weighted_mon_to_year_mean(ssp_msftmz_mon.sel(basin=1, drop=True).sel(lat=26.5, drop=True), "msftmz")
amoc_yr_deep = amoc_yr.sel(lev=slice(500, None))  # depths >= 500 m
max_lev_idx = amoc_yr_deep.idxmax(dim="lev")
ssp_amoc_deep_yr = amoc_yr_deep.sel(lev=max_lev_idx, drop=True) / (1025.0 * 10**6)
#ssp_amoc_yr.to_netcdf(outpath+"ssp_amoc26_yr.nc")

In [ ]:
for sce in ssp_amoc_yr.scenar.values:
    if sce =='ssp585':
        break
    else:
        time = ssp_amoc_yr.sel(scenar=sce).rolling(time=10, center=True).mean("time").time.dt.year
        amoc = ssp_amoc_yr.sel(scenar=sce).rolling(time=10, center=True).mean("time").amoc
        ax1.plot(time, amoc.mean("realiz"), label=sce, alpha=0.8, linewidth=3, color=mpiesm_col[sce])
        ax1.fill_between(time, -amoc.std("realiz")+amoc.mean("realiz"), amoc.std("realiz")+amoc.mean("realiz"), alpha=0.5, color=mpiesm_col[sce])


In [ ]:
fig = plt.figure(figsize=(20, 10))
gs = GridSpec(1, 2)
ax1 = fig.add_subplot(gs[0, 0])
ax2 = fig.add_subplot(gs[0, 1])

for exp in ssphos_amoc26_deep_yr.exper.values:
    time = ssphos_amoc26_deep_yr.sel(exper=exp).rolling(time=10, center=True).mean("time").time.dt.year
    amoc = ssphos_amoc26_deep_yr.sel(exper=exp).rolling(time=10, center=True).mean("time").amoc
    ax1.plot(time, amoc.mean("realiz"), 
             label=exp, alpha=0.8, linewidth=3, color=mpiesm_col[exp])
    
    time = ssphos_amoc_yr.sel(exper=exp).rolling(time=10, center=True).mean("time").time.dt.year
    amoc = ssphos_amoc_yr.sel(exper=exp).rolling(time=10, center=True).mean("time").amoc
    ax2.plot(time, amoc.mean("realiz"), label=exp, alpha=0.8, linewidth=3, color=mpiesm_col[exp])

ax1.set_xlim(2015, 2100)
ax1.set_ylabel(r"$\Delta \mathrm{AMOC}$, 26°N, below 500 m [Sv]")
ax1.spines['right'].set_visible(False)
ax1.spines['top'].set_visible(False)
ax1.spines['left'].set_position(('outward', 10))  # Move the left spine outward by 10 points
ax1.spines['bottom'].set_position(('outward', 10)) 

ax2.set_xlim(2015, 2100)
ax2.set_ylabel(r"$\Delta \mathrm{AMOC}$, 26°N, all depths [Sv]")
ax2.spines['right'].set_visible(False)
ax2.spines['top'].set_visible(False)
ax2.spines['left'].set_position(('outward', 10))  # Move the left spine outward by 10 points
ax2.spines['bottom'].set_position(('outward', 10)) 
#text_legend_hos()
plt.show()

forcings plot

In [ ]:
ssp119 = xr.open_mfdataset("/pool/data/ECHAM6/input/r0008/greenhouse_ssp119.nc")
ssp126 = xr.open_mfdataset("/pool/data/ECHAM6/input/r0008/greenhouse_ssp126.nc")
ssp245 = xr.open_mfdataset("/pool/data/ECHAM6/input/r0008/greenhouse_ssp245.nc")
ssp370 = xr.open_mfdataset("/pool/data/ECHAM6/input/r0008/greenhouse_ssp370.nc")
ssp585 = xr.open_mfdataset("/pool/data/ECHAM6/input/r0008/greenhouse_ssp585.nc")

exps = {
#    "SSP1-1.9": ssp119.isel(lat=0).isel(lon=0),
    "SSP1-2.6": ssp126.isel(lat=0).isel(lon=0),
    "SSP2-4.5": ssp245.isel(lat=0).isel(lon=0),
    "SSP3-7.0": ssp370.isel(lat=0).isel(lon=0),
#    "SSP5-8.5": ssp585.isel(lat=0).isel(lon=0),
}

ssp_col = {#"SSP1-1.9": (0, 173/255, 207/255), 
           "SSP1-2.6": (23/255,60/255,102/255),
           "SSP2-4.5": (247/255,148/255,32/255),
           "SSP3-7.0": (231/255,29/255,37/255),
           #"SSP5-8.5": (149/255, 27/255, 30/255)
          }

In [ ]:
fig, ax = plt.subplots()
for name, exp in exps.items():
    exp.CO2.plot(color=ssp_col[name], label=name, linewidth=5)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.xlim(2014, 2100)
plt.ylim(350, 900)
plt.legend(frameon=False)
plt.title("")
plt.ylabel("CO$_2$ [ppm]")
plt.xlabel("time [year]")
plt.show()
fig.savefig('../plots/co2_forcing.pdf', bbox_inches='tight', transparent=True)

In [ ]:
colors = {
        'neg01': 'midnightblue',
        '01': 'steelblue',
        '03': 'cornflowerblue',
        '05': 'deepskyblue',
        'linneg02': 'teal',
        'lin02': 'cadetblue',
        'lin06': 'mediumaquamarine',
        'lin10': 'aquamarine'
}

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(8, 12), sharex=True)

years = np.linspace(2015, 2101, 85)
ax1.plot(years, np.ones_like(years) * -0.1, linewidth=3, label='-0.1Sv', color=colors['neg01'])
ax1.plot(years, np.ones_like(years) * 0.1, linewidth=3, label='+0.1Sv', color=colors['01'])
ax1.plot(years, np.ones_like(years) * 0.3, linewidth=3, label='+0.3Sv', color=colors['03'])
ax1.plot(years, np.ones_like(years) * 0.5, linewidth=3, label='+0.5Sv', color=colors['05'])
ax1.plot(years, np.linspace(0, -0.2, 85), linewidth=3, label='lin.-0.2Sv', color=colors['linneg02'])
ax1.plot(years, np.linspace(0, 0.2, 85), linewidth=3, label='lin.+0.2Sv', color=colors['lin02'])
ax1.plot(years, np.linspace(0, 0.6, 85), linewidth=3, label='lin.+0.6Sv', color=colors['lin06'])
ax1.plot(years, np.linspace(0, 1.0, 85), linewidth=3, label='lin.+1.0Sv', color=colors['lin10'])
ax1.set_xlim(2015, 2101)
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)
ax1.set_ylabel("Hosing strength [Sv]")
#ax1.legend(frameon=False)
ax1.set_yticks([-0.2, 0, 0.2, 0.4, 0.6, 0.8, 1.0])
ax1.text(-0.15, 1, 'a)', transform=ax1.transAxes, fontsize=18, fontweight='bold', va='top', ha='left')

ax2.plot(years, np.cumsum(np.ones_like(years) * -0.1), linewidth=3, label='-0.1Sv', color=colors['neg01'])
ax2.plot(years, np.cumsum(np.ones_like(years) * 0.1), linewidth=3, label='+0.1Sv', color=colors['01'])
ax2.plot(years, np.cumsum(np.ones_like(years) * 0.3), linewidth=3, label='+0.3Sv', color=colors['03'])
ax2.plot(years, np.cumsum(np.ones_like(years) * 0.5), linewidth=3, label='+0.5Sv', color=colors['05'])
ax2.plot(years, np.cumsum(np.linspace(0, -0.2, 85)), linewidth=3, label='lin.-0.2Sv', color=colors['linneg02'])
ax2.plot(years, np.cumsum(np.linspace(0, 0.2, 85)), linewidth=3, label='lin.+0.2Sv', color=colors['lin02'])
ax2.plot(years, np.cumsum(np.linspace(0, 0.6, 85)), linewidth=3, label='lin.+0.6Sv', color=colors['lin06'])
ax2.plot(years, np.cumsum(np.linspace(0, 1.0, 85)), linewidth=3, label='lin.+1.0Sv', color=colors['lin10'])
ax2.set_xlim(2015, 2101)
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)
ax2.set_ylabel("Cumulative hosing [Sv·yr]")
ax2.set_xlabel("time [year]")
#ax2.legend(frameon=False)
ax2.text(-0.15, 1, 'b)', transform=ax2.transAxes, fontsize=18, fontweight='bold', va='top', ha='left')

handles, labels = ax1.get_legend_handles_labels()
fig.legend(handles, labels, loc='center right', frameon=False, bbox_to_anchor=(1.25, 0.5))
plt.tight_layout()
plt.show()
fig.savefig('../plots/hosing_forcing.pdf', bbox_inches='tight', transparent=True)

ERA5

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 14), gridspec_kw={'hspace': 0.6})

# Top subplot - time series
first_plot = True
for rea in his_tas_yr.realiz:
    ref19601989_gehiseur = weighted_area_lat(his_tas_eur_yr.sel(realiz=rea)).mean('lat').mean('lon').sel(
        time=slice(cftime.DatetimeProlepticGregorian(1960, 1, 1),cftime.DatetimeProlepticGregorian(1989, 12, 31))).mean('time').tas.values
    label = 'MPI-GE historical' if first_plot else None
    (weighted_area_lat(his_tas_eur_yr.sel(realiz=rea)).mean('lat').mean('lon').tas-ref19601989_gehiseur).plot(ax=ax1, color='darkgrey', label=label, linewidth=2)
    first_plot = False
ref19601989_eraeur = weighted_area_lat(era5_tas_eur_yr).mean('lat').mean('lon').sel(
    time=slice(cftime.DatetimeProlepticGregorian(1960, 1, 1),cftime.DatetimeProlepticGregorian(1989, 12, 31))).mean('time').tas.values
(weighted_area_lat(era5_tas_eur_yr).mean('lat').mean('lon').tas-ref19601989_eraeur).plot(ax=ax1, color='royalblue', label='ERA5', linewidth=3)
ax1.set_xlabel("time [year]")
ax1.set_xlim([cftime.DatetimeProlepticGregorian(1940, 1, 1),cftime.DatetimeProlepticGregorian(2014, 1, 31)])
ax1.legend(loc=9, frameon=False)
ax1.set_ylabel(r"$\Delta \mathrm{T}_{\mathrm{EU}}$ [°C]")
ax1.spines['right'].set_visible(False)
ax1.spines['top'].set_visible(False)
ax1.spines['left'].set_position(('outward', 10))
ax1.spines['bottom'].set_position(('outward', 10))

# Bottom subplot - histogram
hisref1960 = weighted_area_lat(his_tas_eur_yr).mean('lat').mean('lon').mean("realiz").sel(time=slice(cftime.DatetimeProlepticGregorian(1960, 1, 1),cftime.DatetimeProlepticGregorian(1989, 12, 31))).mean('time').tas.values
his_tas_eur_yr_anom = weighted_area_lat(his_tas_eur_yr).mean('lat').mean('lon').tas.sel(
        time=slice(cftime.DatetimeProlepticGregorian(1940, 1, 1),cftime.DatetimeProlepticGregorian(2014, 12, 31)))-hisref1960
eraref1960 = weighted_area_lat(era5_tas_eur_yr).mean('lat').mean('lon').sel(time=slice(cftime.DatetimeProlepticGregorian(1960, 1, 1),cftime.DatetimeProlepticGregorian(1989, 12, 31))).mean('time').tas.values
era5_tas_eur_yr_anom = weighted_area_lat(era5_tas_eur_yr).mean('lat').mean('lon').sel(
        time=slice(cftime.DatetimeProlepticGregorian(1940, 1, 1),cftime.DatetimeProlepticGregorian(2014, 12, 31))).tas-eraref1960

ranks_era = np.zeros_like(era5_tas_eur_yr_anom.values)
for i, tas_era in enumerate(era5_tas_eur_yr_anom.values):
    combined = [tas_era]
    for rea in his_tas_eur_yr_anom.realiz:
        combined.append(float(his_tas_eur_yr_anom.sel(realiz=rea).isel(time=i).values))
    ranked = stats.rankdata(combined, method='average')
    ranks_era[i] = ranked[0]

freq_table = np.zeros((50, 50))
rank_tas_realiz = his_tas_eur_yr_anom.rank(dim='realiz')
for i, rea in enumerate(his_tas_eur_yr_anom.realiz):
    freq, bin_edges = np.histogram(rank_tas_realiz.sel(realiz=rea), bins=np.arange(1, 52)-0.5)
    freq_table[i, :] = freq
    
freq_model10 = []
freq_model90 = []
for j, rea in enumerate(his_tas_eur_yr_anom.realiz):
    freq_model10.append(np.quantile(freq_table[:, j], 0.1))
    freq_model90.append(np.quantile(freq_table[:, j], 0.9))

hist, bin_edges = np.histogram(ranks_era, bins=np.arange(1, 52)-0.5)
ax2.hist(ranks_era, bins=np.arange(52)-0.5, alpha=0.75, label='ERA5 histograms', color='skyblue')

mean_freq = np.convolve(hist, np.ones(7)/7, mode='valid')
ax2.plot(np.arange(4, 48), mean_freq, alpha=0.85, color='royalblue', label='ERA5 7-bin window histogram slope', linewidth=4)

ax2.plot(np.arange(4, 48), np.convolve(freq_model10, np.ones(7)/7, mode='valid'), 
         linestyle='dashed', color='black', linewidth=4)
ax2.plot(np.arange(4, 48), np.convolve(freq_model90, np.ones(7)/7, mode='valid'),
         linestyle='dashed', color='black', linewidth=4, label='Perfect model 7-bin window histogram slope range (central p80)')
ax2.set_xlabel('Rank')
ax2.set_ylabel('Frequency')
ax2.set_xlim(0, 51)
ax2.legend(loc=9, bbox_to_anchor=(0.48, 1.2), frameon=False)
ax2.spines['right'].set_visible(False)
ax2.spines['top'].set_visible(False)
ax2.spines['left'].set_position(('outward', 10))
ax2.spines['bottom'].set_position(('outward', 10))

plt.tight_layout()
plt.show()
fig.savefig('../plots/rank_histograms.pdf', bbox_inches='tight', transparent=True)

In [ ]:
# Annual mean T
models = ["CanESM5", "CESM2", "EC-Earth3", "HadGEM3-GC3-1LL", "HadGEM3-GC3-1MM", "IPSL-CM6A-LR", "MPI-ESM1-2-HR", "MPI-ESM1-2-LR"]
path = "/work/uo1075/m300817/hosing/nahosmip/" 
siconc_nahosmip = {}
for model in models:
    ds =  xr.open_mfdataset(path+f"/{model}/u03-hos/*siconc*.nc", use_cftime=True, parallel=True)
    if 'lon' in ds.coords:
        lon_coord = 'lon'
    elif 'longitude' in ds.coords:
        lon_coord = 'longitude'
    else:
        # Handle case where longitude coordinate has a different name
        lon_coord = None
        for coord in ds.coords:
            if 'lon' in coord.lower():
                lon_coord = coord
                break
    
    if lon_coord and ds[lon_coord].max() > 180:
        # Convert from 0-360 to -180-180
        ds = ds.assign_coords({lon_coord: (ds[lon_coord] + 180) % 360 - 180})
        ds = ds.sortby(lon_coord)
    
    siconc_nahosmip[model] = ds

In [ ]:
possible_var_names = ['siconc', 'aice', 'soicecov']   # Define possible variable names for sea ice concentration
possible_time_names = ['time', 'time_counter']   # Define possible time dimension names

def get_siconc_variable(ds, possible_names):
    """Find the sea ice concentration variable in the dataset"""
    for var_name in possible_names:
        if var_name in ds.variables:
            return var_name
    
    # If none found, look for variables with 'ice' and 'conc' in the name
    for var_name in ds.variables:
        if 'ice' in var_name.lower() and 'conc' in var_name.lower():
            return var_name
    
    # If still nothing, look for any variable with 'ice' in the name
    for var_name in ds.variables:
        if 'ice' in var_name.lower():
            return var_name
    
    # Last resort: print available variables and return None
    print(f"Could not find sea ice concentration variable. Available variables: {list(ds.variables.keys())}")
    return None

def get_time_dimension(ds, possible_names):
    """Find the time dimension in the dataset"""
    for time_name in possible_names:
        if time_name in ds.dims:
            return time_name
    
    # If none found, look for dimensions with 'time' in the name
    for dim_name in ds.dims:
        if 'time' in dim_name.lower():
            return dim_name
    
    # Last resort: print available dimensions and return None
    print(f"Could not find time dimension. Available dimensions: {list(ds.dims.keys())}")
    return None

In [ ]:
# Define North Atlantic Arctic region bounds
lat_min, lat_max = 40, 90
lon_min, lon_max = -80, 55

# Create subplot figure
fig = plt.figure(figsize=(20, 13))
n_cols = 3
n_rows = 3
for i, model in enumerate(models):
    ds = siconc_nahosmip[model]
    var_name = get_siconc_variable(ds, possible_var_names)
    time_dim = get_time_dimension(ds, possible_time_names)
    ds_region = ds.sel(lat=slice(lat_min, lat_max), lon=slice(lon_min, lon_max))
    ds_last5 = ds_region[var_name].isel({time_dim: slice(95, 100)}).mean(dim=time_dim)
    ds_first5 = ds_region[var_name].isel({time_dim: slice(0, 5)}).mean(dim=time_dim)
    year = 99
    siconc_plot = ds_region[var_name].isel({time_dim: year})
    
    ax = fig.add_subplot(n_rows, n_cols, i+1, projection=ccrs.PlateCarree())

    im = siconc_plot.plot(ax=ax, transform=ccrs.PlateCarree(), cmap='Blues_r', 
                          vmin=0, vmax=1, levels=np.linspace(0.,1,21), add_colorbar=False)
    
    #ax.coastlines()
    ax.add_feature(cfeature.LAND, color='lightgray')
    ax.add_feature(cfeature.OCEAN, color='white')
    
    ax.set_extent([lon_min, lon_max, lat_min, lat_max], ccrs.PlateCarree())
    ax.set_title(f'{model}', fontsize=14, y=0.95)
    ax.set_frame_on(False)
cbar_ax = fig.add_axes([0.92, 0.2, 0.02, 0.6])
cbar = fig.colorbar(im, cax=cbar_ax, ticks=[0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1])
cbar.set_label('Sea ice concentration', rotation=270, labelpad=20)

plt.tight_layout()
plt.subplots_adjust(hspace=-0.61, right=0.9)
plt.show()
fig.savefig('../plots/seaiceconc_100y.pdf', bbox_inches='tight', transparent=True)